======================================================================================================

초기환경 설정

======================================================================================================

In [ ]:
!pip install torch torchvision torchaudio

In [ ]:
!pip uninstall torch torchvision torchaudio -y

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [ ]:
!pip install tqdm

In [ ]:
import sys
import torch

print(f"현재 사용 중인 파이썬 경로: {sys.executable}") 
# 결과에 'brain_ai' 단어가 포함되어 있어야 합니다.

print(f"PyTorch 버전: {torch.__version__}")
print(f"GPU 가속 가능 여부: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"현재 잡혀있는 GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
import os
import glob

def count_dataset(root_path):
    for split in ['TRAIN', 'TEST','VAL']:
        split_path = os.path.join(root_path, split)
        if not os.path.exists(split_path): continue
        
        all_pre = glob.glob(f"{split_path}/*/pre/*")
        all_post = glob.glob(f"{split_path}/*/post/*")
        all_mask = glob.glob(f"{split_path}/*/mask/*")
        patients = os.listdir(split_path)
        
        print(f"--- {split} Statistics ---")
        print(f"총 환자 수: {len(patients)}명")
        print(f"Total PRE (Input): {len(all_pre)}장")
        print(f"Total POST (Target): {len(all_post)}장")
        print(f"Total MASK (GT): {len(all_mask)}장")
        print(f"총 합계(파일 수): {len(all_pre) + len(all_post) + len(all_mask)}장\n")
    print(17796+4770+5220)

# 실행
count_dataset("./pix2pix_curated_withMask_1-100")

In [ ]:
import os
import shutil

# =========================
# 경로 설정
# =========================
SRC1 = "./pix2pix_curated_withMask"
SRC2 = "./pix2pix_curated_withMask_51-100"
DST  = "./pix2pix_curated_withMask_1-100"

SPLITS = ["TRAIN", "VAL", "TEST"]


# =========================
# 함수: 폴더 복사
# =========================
def copy_dataset(src, dst):
    for split in SPLITS:
        src_split = os.path.join(src, split)
        dst_split = os.path.join(dst, split)

        os.makedirs(dst_split, exist_ok=True)

        for patient in os.listdir(src_split):
            src_path = os.path.join(src_split, patient)
            dst_path = os.path.join(dst_split, patient)

            if os.path.exists(dst_path):
                print(f" 중복 환자 스킵: {patient}")
                continue

            shutil.copytree(src_path, dst_path)


# =========================
# 실행
# =========================
print("dataset 1 복사 중...")
copy_dataset(SRC1, DST)

print("dataset 2 복사 중...")
copy_dataset(SRC2, DST)


# =========================
# 결과 확인
# =========================
print("\n===== 결과 =====")
for split in SPLITS:
    path = os.path.join(DST, split)
    count = len(os.listdir(path))
    print(f"{split}: {count}명")

print("\n완료: 100명 dataset 생성됨")

======================================================================================================

 BASIC_Pix2Pix 모델 학습
 
======================================================================================================

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
from tqdm import tqdm

# =========================
# 1. CONFIG
# =========================
DATA_ROOT = "./pix2pix_curated"
CHECKPOINT_DIR = "./pix2pix_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

BATCH_SIZE = 4
EPOCHS = 300
LR_G = 2e-4
LR_D = 2e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LAMBDA_L1 = 100

# =========================
# 2. DATASET
# =========================
class Pix2PixDataset(Dataset):
    def __init__(self, root):
        self.pairs = []
        self.tf = T.Compose([
            T.Resize((256, 256)),
            T.ToTensor(),
            T.Normalize((0.5,), (0.5,))
        ])

        if not os.path.isdir(root):
            raise FileNotFoundError(f"데이터셋 루트 디렉토리를 찾을 수 없습니다: {root}")

        for pid in sorted(os.listdir(root)):
            pre_dir = os.path.join(root, pid, "pre")
            post_dir = os.path.join(root, pid, "post")
            if not os.path.isdir(pre_dir) or not os.path.isdir(post_dir):
                continue

            for f in sorted(os.listdir(pre_dir)):
                pre_path = os.path.join(pre_dir, f)
                post_path = os.path.join(post_dir, f)
                if os.path.exists(post_path):
                    self.pairs.append((pre_path, post_path))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        pre_path, post_path = self.pairs[idx]
        pre_img = Image.open(pre_path).convert("L")
        post_img = Image.open(post_path).convert("L")
        pre_tensor = self.tf(pre_img)
        post_tensor = self.tf(post_img)
        return pre_tensor, post_tensor

# =========================
# 3. GENERATOR & 4. DISCRIMINATOR (구조는 동일)
# =========================
class UNetBlock(nn.Module):
    def __init__(self, in_ch, out_ch, submodule=None, innermost=False, outermost=False):
        super().__init__()
        self.outermost = outermost
        downconv = nn.Conv2d(in_ch, out_ch, 4, 2, 1, bias=False)
        downrelu = nn.LeakyReLU(0.2, True)
        downnorm = nn.BatchNorm2d(out_ch)
        uprelu = nn.ReLU(True)

        if outermost:
            upconv = nn.ConvTranspose2d(out_ch * 2, in_ch, 4, 2, 1)
            model = [downconv] + [submodule] + [uprelu, upconv, nn.Tanh()]
        elif innermost:
            upconv = nn.ConvTranspose2d(out_ch, in_ch, 4, 2, 1, bias=False)
            model = [downrelu, downconv] + [uprelu, upconv, nn.BatchNorm2d(in_ch)]
        else:
            upconv = nn.ConvTranspose2d(out_ch * 2, in_ch, 4, 2, 1, bias=False)
            model = [downrelu, downconv, downnorm] + [submodule] + [uprelu, upconv, nn.BatchNorm2d(in_ch)]

        self.model = nn.Sequential(*model)

    def forward(self, x):
        if self.outermost: return self.model(x)
        else: return torch.cat([x, self.model(x)], 1)

class UNetGenerator(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, num_filters=64):
        super().__init__()
        b8 = UNetBlock(num_filters*8, num_filters*8, innermost=True)
        b7 = UNetBlock(num_filters*8, num_filters*8, submodule=b8)
        b6 = UNetBlock(num_filters*8, num_filters*8, submodule=b7)
        b5 = UNetBlock(num_filters*8, num_filters*8, submodule=b6)
        b4 = UNetBlock(num_filters*4, num_filters*8, submodule=b5)
        b3 = UNetBlock(num_filters*2, num_filters*4, submodule=b4)
        b2 = UNetBlock(num_filters, num_filters*2, submodule=b3)
        self.model = UNetBlock(out_ch, num_filters, submodule=b2, outermost=True)
    def forward(self, x): return self.model(x)

class PatchDiscriminator(nn.Module):
    def __init__(self, in_ch=2):
        super().__init__()
        def block(i, o, s=2):
            return nn.Sequential(
                nn.Conv2d(i, o, 4, s, 1, bias=False),
                nn.BatchNorm2d(o),
                nn.LeakyReLU(0.2, inplace=True)
            )
        self.model = nn.Sequential(
            nn.Conv2d(in_ch, 64, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            block(64, 128),
            block(128, 256),
            block(256, 512, s=1),
            nn.Conv2d(512, 1, 4, 1, 1)
        )
    def forward(self, x, y): return self.model(torch.cat([x, y], 1))

# =========================
# 5. INITIALIZATION & TRAINING
# =========================
train_loader = DataLoader(Pix2PixDataset(f"{DATA_ROOT}/TRAIN"), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(Pix2PixDataset(f"{DATA_ROOT}/VAL"), batch_size=BATCH_SIZE, shuffle=False)

G = UNetGenerator(in_ch=1, out_ch=1).to(DEVICE)
D = PatchDiscriminator(in_ch=2).to(DEVICE)

opt_G = optim.Adam(G.parameters(), lr=LR_G, betas=(0.5, 0.999))
opt_D = optim.Adam(D.parameters(), lr=LR_D, betas=(0.5, 0.999))

criterion_GAN = nn.BCEWithLogitsLoss()
criterion_L1 = nn.L1Loss()

print("Training Start...")
for epoch in range(EPOCHS):
    # --- Train Phase ---
    G.train(); D.train()
    train_loop = tqdm(train_loader, leave=False, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    for pre, post in train_loop:
        pre, post = pre.to(DEVICE), post.to(DEVICE)

        fake = G(pre)
        real_pred = D(pre, post)
        fake_pred = D(pre, fake.detach())
        loss_D = (criterion_GAN(real_pred, torch.ones_like(real_pred)) +
                  criterion_GAN(fake_pred, torch.zeros_like(fake_pred))) * 0.5

        opt_D.zero_grad()
        loss_D.backward()
        opt_D.step()

        fake_pred = D(pre, fake)
        loss_G_GAN = criterion_GAN(fake_pred, torch.ones_like(fake_pred))
        loss_G_L1 = criterion_L1(fake, post) * LAMBDA_L1
        loss_G = loss_G_GAN + loss_G_L1

        opt_G.zero_grad()
        loss_G.backward()
        opt_G.step()

        train_loop.set_postfix(G_loss=f"{loss_G.item():.4f}", D_loss=f"{loss_D.item():.4f}")

    # --- Validation Phase ---
    G.eval(); D.eval()
    val_g_loss_total = 0.0
    val_samples = 0
    with torch.no_grad():
        for pre_val, post_val in val_loader:
            pre_val, post_val = pre_val.to(DEVICE), post_val.to(DEVICE)
            fake_val = G(pre_val)
            loss_G_L1_val = criterion_L1(fake_val, post_val) * LAMBDA_L1
            val_g_loss_total += loss_G_L1_val.item() * pre_val.size(0)
            val_samples += pre_val.size(0)
    avg_val_g_loss = val_g_loss_total / val_samples if val_samples > 0 else 0

    print(f"Epoch [{epoch+1}/{EPOCHS}] Train G_loss: {loss_G.item():.4f} | Val L1_loss: {avg_val_g_loss:.4f}")

    # 공통 저장 데이터
    checkpoint_data = {
        'epoch': epoch + 1,
        'G_state': G.state_dict(),
        'D_state': D.state_dict(),
        'opt_G_state': opt_G.state_dict(),
        'opt_D_state': opt_D.state_dict(),
        'val_g_loss': avg_val_g_loss
    }

    # [1] 매 에포크마다 최신 모델 덮어쓰기 저장
    torch.save(checkpoint_data, f"{CHECKPOINT_DIR}/latest_checkpoint.pt")

    # [2] 50 에포크마다 별도 파일로 정기 저장
    if (epoch + 1) % 50 == 0:
        torch.save(checkpoint_data, f"{CHECKPOINT_DIR}/checkpoint_epoch_{epoch+1}.pt")
        print(f"--- 정기 체크포인트 저장 완료 (Epoch {epoch+1}) ---")

print("Training Finished.")

-------------------------------------------------------------------------------

++이어서 학습++

-------------------------------------------------------------------------------

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
from tqdm import tqdm

# ==========================
# 설정
# ==========================
DATA_ROOT = "./pix2pix_curated"  # 데이터셋 루트
CHECKPOINT_DIR = "./pix2pix_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "checkpoint_epoch_110.pt")

BATCH_SIZE = 4
ADDITIONAL_EPOCHS = 200
LR_G = 2e-4
LR_D = 2e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LAMBDA_L1 = 100

# ==========================
# Dataset 정의
# ==========================
class Pix2PixDataset(Dataset):
    def __init__(self, root):
        self.pairs = []
        self.tf = T.Compose([T.Resize((256, 256)), T.ToTensor(), T.Normalize((0.5,), (0.5,))])

        if not os.path.isdir(root):
            raise FileNotFoundError(f"데이터셋 루트 디렉토리를 찾을 수 없습니다: {root}")

        for pid in sorted(os.listdir(root)):
            pre_dir, post_dir = os.path.join(root, pid, "pre"), os.path.join(root, pid, "post")
            if not os.path.isdir(pre_dir) or not os.path.isdir(post_dir):
                continue
            for f in sorted(os.listdir(pre_dir)):
                pre_path = os.path.join(pre_dir, f)
                post_path = os.path.join(post_dir, f)
                if os.path.exists(post_path):
                    self.pairs.append((pre_path, post_path))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        pre = self.tf(Image.open(self.pairs[idx][0]).convert("L"))
        post = self.tf(Image.open(self.pairs[idx][1]).convert("L"))
        return pre, post

# ==========================
# 모델 정의
# ==========================
class UNetBlock(nn.Module):
    def __init__(self, in_ch, out_ch, submodule=None, innermost=False, outermost=False):
        super().__init__()
        self.outermost = outermost
        downconv = nn.Conv2d(in_ch, out_ch, 4, 2, 1, bias=False)
        downrelu, downnorm = nn.LeakyReLU(0.2, True), nn.BatchNorm2d(out_ch)
        uprelu, upnorm = nn.ReLU(True), nn.BatchNorm2d(in_ch)

        if outermost:
            upconv = nn.ConvTranspose2d(out_ch * 2, in_ch, 4, 2, 1)
            model = [downconv, submodule, uprelu, upconv, nn.Tanh()]
        elif innermost:
            upconv = nn.ConvTranspose2d(out_ch, in_ch, 4, 2, 1, bias=False)
            model = [downrelu, downconv, uprelu, upconv, upnorm]
        else:
            upconv = nn.ConvTranspose2d(out_ch * 2, in_ch, 4, 2, 1, bias=False)
            model = [downrelu, downconv, downnorm, submodule, uprelu, upconv, upnorm]

        self.model = nn.Sequential(*model)

    def forward(self, x):
        if self.outermost:
            return self.model(x)
        else:
            return torch.cat([x, self.model(x)], 1)

class UNetGenerator(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, num_filters=64):
        super().__init__()
        b8 = UNetBlock(num_filters*8, num_filters*8, innermost=True)
        b7 = UNetBlock(num_filters*8, num_filters*8, submodule=b8)
        b6 = UNetBlock(num_filters*8, num_filters*8, submodule=b7)
        b5 = UNetBlock(num_filters*8, num_filters*8, submodule=b6)
        b4 = UNetBlock(num_filters*4, num_filters*8, submodule=b5)
        b3 = UNetBlock(num_filters*2, num_filters*4, submodule=b4)
        b2 = UNetBlock(num_filters, num_filters*2, submodule=b3)
        self.model = UNetBlock(out_ch, num_filters, submodule=b2, outermost=True)

    def forward(self, x):
        return self.model(x)

class PatchDiscriminator(nn.Module):
    def __init__(self, in_ch=2):
        super().__init__()
        def block(i, o, s=2):
            return nn.Sequential(
                nn.Conv2d(i, o, 4, s, 1, bias=False),
                nn.BatchNorm2d(o),
                nn.LeakyReLU(0.2, inplace=True)
            )
        self.model = nn.Sequential(
            nn.Conv2d(in_ch, 64, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            block(64, 128),
            block(128, 256),
            block(256, 512, s=1),
            nn.Conv2d(512, 1, 4, 1, 1)
        )

    def forward(self, x, y):
        return self.model(torch.cat([x, y], 1))

# ==========================
# 데이터로더 초기화
# ==========================
train_loader = DataLoader(Pix2PixDataset(f"{DATA_ROOT}/TRAIN"), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(Pix2PixDataset(f"{DATA_ROOT}/VAL"), batch_size=BATCH_SIZE, shuffle=False)

# ==========================
# 모델, Optimizer, Loss
# ==========================
G = UNetGenerator(in_ch=1, out_ch=1).to(DEVICE)
D = PatchDiscriminator(in_ch=2).to(DEVICE)
opt_G = optim.Adam(G.parameters(), lr=LR_G, betas=(0.5, 0.999))
opt_D = optim.Adam(D.parameters(), lr=LR_D, betas=(0.5, 0.999))
criterion_GAN, criterion_L1 = nn.BCEWithLogitsLoss(), nn.L1Loss()

# ==========================
# 체크포인트 로드
# ==========================
start_epoch = 0
best_val_loss = float('inf')

if os.path.exists(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    G.load_state_dict(ckpt['G_state'])
    D.load_state_dict(ckpt['D_state'])
    opt_G.load_state_dict(ckpt['opt_G_state'])
    opt_D.load_state_dict(ckpt['opt_D_state'])
    start_epoch = ckpt['epoch'] + 1
    best_val_loss = ckpt.get('best_val_g_loss', best_val_loss)
    print(f"Resuming from epoch {start_epoch} with best_val_g_loss: {best_val_loss:.4f}")
else:
    print("No checkpoint found. Starting from scratch.")

# ==========================
# 학습 루프
# ==========================
print("Training Start...")
for epoch in range(start_epoch, start_epoch + ADDITIONAL_EPOCHS):
    # --- Train ---
    G.train(); D.train()
    train_loop = tqdm(train_loader, leave=False, desc=f"Epoch {epoch+1}/{start_epoch + ADDITIONAL_EPOCHS} [Train]")
    for pre, post in train_loop:
        pre, post = pre.to(DEVICE), post.to(DEVICE)

        # Discriminator
        fake = G(pre)
        real_pred = D(pre, post)
        fake_pred = D(pre, fake.detach())
        loss_D = (criterion_GAN(real_pred, torch.ones_like(real_pred)) +
                  criterion_GAN(fake_pred, torch.zeros_like(fake_pred))) * 0.5
        opt_D.zero_grad()
        loss_D.backward()
        opt_D.step()

        # Generator
        fake_pred = D(pre, fake)
        loss_G_GAN = criterion_GAN(fake_pred, torch.ones_like(fake_pred))
        loss_G_L1 = criterion_L1(fake, post) * LAMBDA_L1
        loss_G = loss_G_GAN + loss_G_L1
        opt_G.zero_grad()
        loss_G.backward()
        opt_G.step()

        train_loop.set_postfix(G_loss=f"{loss_G.item():.4f}", D_loss=f"{loss_D.item():.4f}")

    # --- Validation ---
    G.eval(); D.eval()
    val_g_loss_total = 0.0
    val_d_loss_total = 0.0
    val_samples = 0
    val_loop = tqdm(val_loader, leave=False, desc=f"Epoch {epoch+1}/{start_epoch + ADDITIONAL_EPOCHS} [Val]")
    with torch.no_grad():
        for pre_val, post_val in val_loop:
            pre_val, post_val = pre_val.to(DEVICE), post_val.to(DEVICE)
            fake_val = G(pre_val)
            fake_pred_val = D(pre_val, fake_val)
            loss_G_val = criterion_GAN(fake_pred_val, torch.ones_like(fake_pred_val)) + criterion_L1(fake_val, post_val) * LAMBDA_L1
            val_g_loss_total += loss_G_val.item() * pre_val.size(0)

            real_pred_val = D(pre_val, post_val)
            fake_pred_val_detach = D(pre_val, fake_val.detach())
            loss_D_val = (criterion_GAN(real_pred_val, torch.ones_like(real_pred_val)) +
                          criterion_GAN(fake_pred_val_detach, torch.zeros_like(fake_pred_val_detach))) * 0.5
            val_d_loss_total += loss_D_val.item() * pre_val.size(0)
            val_samples += pre_val.size(0)
            val_loop.set_postfix(G_loss=f"{loss_G_val.item():.4f}", D_loss=f"{loss_D_val.item():.4f}")

    avg_val_g_loss = val_g_loss_total / val_samples if val_samples > 0 else 0
    avg_val_d_loss = val_d_loss_total / val_samples if val_samples > 0 else 0

    print(f"Epoch [{epoch+1}/{start_epoch + ADDITIONAL_EPOCHS}] Train G_loss: {loss_G.item():.4f}, D_loss: {loss_D.item():.4f} | Val G_loss: {avg_val_g_loss:.4f}, D_loss: {avg_val_d_loss:.4f}")

    # ==========================
    # --- 체크포인트 저장 ---
    # ==========================
    
    # 1) latest 체크포인트는 매 에포크마다 갱신 (기존 파일 덮어쓰기)
    latest_checkpoint_path = os.path.join(CHECKPOINT_DIR, "checkpoint_latest.pt")
    torch.save({
        'epoch': epoch,
        'G_state': G.state_dict(),
        'D_state': D.state_dict(),
        'opt_G_state': opt_G.state_dict(),
        'opt_D_state': opt_D.state_dict(),
        'best_val_g_loss': best_val_loss
    }, latest_checkpoint_path)
    print(f"[Checkpoint] Latest saved at epoch {epoch+1}")
    
    # 2) 50 에포크마다 별도 저장
    if (epoch + 1) % 50 == 0:
        checkpoint_50_path = os.path.join(CHECKPOINT_DIR, f"checkpoint_epoch_{epoch+1}.pt")
        torch.save({
            'epoch': epoch,
            'G_state': G.state_dict(),
            'D_state': D.state_dict(),
            'opt_G_state': opt_G.state_dict(),
            'opt_D_state': opt_D.state_dict(),
            'best_val_g_loss': best_val_loss
        }, checkpoint_50_path)
        print(f"[Checkpoint] Saved checkpoint_epoch_{epoch+1}.pt")

print("Training Finished.")

-------------------------------------------------------------------------------

모델 생성 테스트

-------------------------------------------------------------------------------

In [ ]:
import os
import torch
import torch.nn as nn
from PIL import Image
import torchvision.transforms as T
import matplotlib.pyplot as plt

# =========================
# 1. 설정 (CONFIG)
# =========================
# DATA_ROOT 경로를 생성된 데이터셋의 'TEST' 폴더로 변경하고, 존재하는 환자 ID를 지정
DATA_ROOT = "./pix2pix_curated/TEST" # test 폴더 경로 수정
TARGET_PATIENT = "100126B" # TOP_50_PATIENTS 중 'TEST' 세트에 있는 환자 ID로 변경 (예: 100285A는 인덱스 5, 즉 rem=5로 TEST에 해당)
MODEL_PATH = "./pix2pix_checkpoints/checkpoint_epoch_200.pt"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# [모델 정의: UNetBlock, UNetGenerator는 동일]
class UNetBlock(nn.Module):
    def __init__(self, in_ch, out_ch, submodule=None, innermost=False, outermost=False):
        super().__init__()
        self.outermost = outermost
        downconv = nn.Conv2d(in_ch, out_ch, 4, 2, 1, bias=False)
        downrelu, downnorm = nn.LeakyReLU(0.2, True), nn.BatchNorm2d(out_ch)
        uprelu, upnorm = nn.ReLU(True), nn.BatchNorm2d(in_ch)
        if outermost:
            upconv = nn.ConvTranspose2d(out_ch * 2, in_ch, 4, 2, 1)
            model = [downconv, submodule, uprelu, upconv, nn.Tanh()]
        elif innermost:
            upconv = nn.ConvTranspose2d(out_ch, in_ch, 4, 2, 1, bias=False)
            model = [downrelu, downconv, uprelu, upconv, upnorm]
        else:
            upconv = nn.ConvTranspose2d(out_ch * 2, in_ch, 4, 2, 1, bias=False)
            model = [downrelu, downconv, downnorm, submodule, uprelu, upconv, upnorm]
        self.model = nn.Sequential(*model)
    def forward(self, x): return self.model(x) if self.outermost else torch.cat([x, self.model(x)], 1)

class UNetGenerator(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, num_filters=64):
        super().__init__()
        # 8-layer UNet
        b8 = UNetBlock(num_filters*8, num_filters*8, innermost=True)
        b7 = UNetBlock(num_filters*8, num_filters*8, submodule=b8)
        b6 = UNetBlock(num_filters*8, num_filters*8, submodule=b7)
        b5 = UNetBlock(num_filters*8, num_filters*8, submodule=b6)
        b4 = UNetBlock(num_filters*4, num_filters*8, submodule=b5)
        b3 = UNetBlock(num_filters*2, num_filters*4, submodule=b4)
        b2 = UNetBlock(num_filters, num_filters*2, submodule=b3)
        self.model = UNetBlock(out_ch, num_filters, submodule=b2, outermost=True)
    def forward(self, x): return self.model(x)

# =========================
# 2. 실행 루틴
# =========================
G = UNetGenerator(in_ch=1, out_ch=1).to(DEVICE) # Generator 초기화 시 채널 인자 명시
ckpt = torch.load(MODEL_PATH, map_location=DEVICE)
G.load_state_dict(ckpt['G_state'] if 'G_state' in ckpt else ckpt)
G.eval()

# 전처리 설정
tf = T.Compose([T.Resize((256, 256)), T.ToTensor(), T.Normalize((0.5,), (0.5,))])
rescale = lambda x: (x + 1.0) / 2.0

# 환자 경로 설정
patient_base_path = os.path.join(DATA_ROOT, TARGET_PATIENT)
pre_path = os.path.join(patient_base_path, "pre")
post_path = os.path.join(patient_base_path, "post")

if not os.path.exists(pre_path):
    print(f"경로를 찾을 수 없습니다: {pre_path}. TARGET_PATIENT가 올바른 데이터셋 분할에 있는지 확인해주세요.")
else:
    file_list = sorted(os.listdir(pre_path))
    print(f"환자 '{TARGET_PATIENT}'의 이미지 {len(file_list)}장을 분석합니다...")

    with torch.no_grad():
        for filename in file_list:
            # 이미지 로드 (강제 1채널 흑백)
            img_pre = Image.open(os.path.join(pre_path, filename)).convert("L")
            img_post = Image.open(os.path.join(post_path, filename)).convert("L")

            # 텐서 변환 및 모델 입력
            input_tensor = tf(img_pre).unsqueeze(0).to(DEVICE) # Batch 차원 추가
            fake_tensor = G(input_tensor).cpu().squeeze(0)

            # 시각화 (Input | Fake | Target)
            plt.figure(figsize=(15, 5))

            plt.subplot(1, 3, 1)
            plt.imshow(rescale(input_tensor.cpu().squeeze()), cmap='gray')
            plt.title(f"Pre (Input)\n{filename}")
            plt.axis('off')

            plt.subplot(1, 3, 2)
            plt.imshow(rescale(fake_tensor[0]), cmap='gray')
            plt.title("Generated (Fake Post)")
            plt.axis('off')

            plt.subplot(1, 3, 3)
            plt.imshow(rescale(tf(img_post).squeeze()), cmap='gray')
            plt.title("Real (Target Post)")
            plt.axis('off')

            plt.tight_layout()
            plt.show()

print("--- 분석 완료 ---")

In [ ]:
import os
import torch
import torch.nn as nn
from PIL import Image
import torchvision.transforms as T
import cv2
import numpy as np

# =========================
# 설정
# =========================
DATA_ROOT = "./pix2pix_curated/TEST"
TARGET_PATIENT = "100126B"
MODEL_PATH = "./pix2pix_checkpoints/checkpoint_epoch_200.pt"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

SAVE_DIR = "./results"
os.makedirs(SAVE_DIR, exist_ok=True)

PAD = 5


# =========================
# 모델 정의
# =========================
class UNetBlock(nn.Module):
    def __init__(self, in_ch, out_ch, submodule=None, innermost=False, outermost=False):
        super().__init__()
        self.outermost = outermost
        downconv = nn.Conv2d(in_ch, out_ch, 4, 2, 1, bias=False)
        downrelu, downnorm = nn.LeakyReLU(0.2, True), nn.BatchNorm2d(out_ch)
        uprelu, upnorm = nn.ReLU(True), nn.BatchNorm2d(in_ch)

        if outermost:
            upconv = nn.ConvTranspose2d(out_ch * 2, in_ch, 4, 2, 1)
            model = [downconv, submodule, uprelu, upconv, nn.Tanh()]
        elif innermost:
            upconv = nn.ConvTranspose2d(out_ch, in_ch, 4, 2, 1, bias=False)
            model = [downrelu, downconv, uprelu, upconv, upnorm]
        else:
            upconv = nn.ConvTranspose2d(out_ch * 2, in_ch, 4, 2, 1, bias=False)
            model = [downrelu, downconv, downnorm, submodule, uprelu, upconv, upnorm]

        self.model = nn.Sequential(*model)

    def forward(self, x):
        return self.model(x) if self.outermost else torch.cat([x, self.model(x)], 1)


class UNetGenerator(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, num_filters=64):
        super().__init__()
        b8 = UNetBlock(num_filters*8, num_filters*8, innermost=True)
        b7 = UNetBlock(num_filters*8, num_filters*8, submodule=b8)
        b6 = UNetBlock(num_filters*8, num_filters*8, submodule=b7)
        b5 = UNetBlock(num_filters*8, num_filters*8, submodule=b6)
        b4 = UNetBlock(num_filters*4, num_filters*8, submodule=b5)
        b3 = UNetBlock(num_filters*2, num_filters*4, submodule=b4)
        b2 = UNetBlock(num_filters, num_filters*2, submodule=b3)
        self.model = UNetBlock(out_ch, num_filters, submodule=b2, outermost=True)

    def forward(self, x):
        return self.model(x)


# =========================
# 모델 로드
# =========================
G = UNetGenerator().to(DEVICE)
ckpt = torch.load(MODEL_PATH, map_location=DEVICE)
G.load_state_dict(ckpt['G_state'] if 'G_state' in ckpt else ckpt)
G.eval()

tf = T.Compose([
    T.Resize((256, 256)),
    T.ToTensor(),
    T.Normalize((0.5,), (0.5,))
])

rescale = lambda x: (x + 1.0) / 2.0


# =========================
# 데이터 경로
# =========================
patient_base_path = os.path.join(DATA_ROOT, TARGET_PATIENT)
pre_path = os.path.join(patient_base_path, "pre")
post_path = os.path.join(patient_base_path, "post")

all_rows = []

# =========================
# 실행
# =========================
file_list = sorted(os.listdir(pre_path))

with torch.no_grad():
    for i, filename in enumerate(file_list):

        img_pre = Image.open(os.path.join(pre_path, filename)).convert("L")
        img_post = Image.open(os.path.join(post_path, filename)).convert("L")

        input_tensor = tf(img_pre).unsqueeze(0).to(DEVICE)
        fake_tensor = G(input_tensor).cpu().squeeze(0)

        pre_img = rescale(input_tensor.cpu().squeeze()).numpy()
        fake_img = rescale(fake_tensor[0]).numpy()
        post_img = rescale(tf(img_post).squeeze()).numpy()

        h, w = pre_img.shape

        # =========================
        # 가로 padding
        # =========================
        v_pad = np.ones((h, PAD))

        combined = np.concatenate([
            pre_img, v_pad,
            fake_img, v_pad,
            post_img
        ], axis=1)

        combined = (combined * 255).astype(np.uint8)
        combined = cv2.cvtColor(combined, cv2.COLOR_GRAY2BGR)

        # =========================
        # 텍스트
        # =========================
        cv2.putText(combined, f"Slice {i}", (10, 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)

        labels = ["PRE", "FAKE_POST", "POST"]
        step = w + PAD

        for idx, label in enumerate(labels):
            x = idx * step + 5
            cv2.putText(combined, label, (x, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,200,0), 1)

        all_rows.append(combined)


# =========================
# 세로로 이어붙이기
# =========================
row_h, row_w, _ = all_rows[0].shape
h_pad = np.ones((PAD, row_w, 3), dtype=np.uint8) * 255

final_list = []
for r in all_rows:
    final_list.append(r)
    final_list.append(h_pad)

final_img = np.concatenate(final_list[:-1], axis=0)

# =========================
# 최종 저장
# =========================
cv2.imwrite(os.path.join(SAVE_DIR, "final_result_nomask.png"), final_img)

print("완료: final_result_nomask.png 저장됨")

-------------------------------------------------------------------------------
SSIM 지표 테스트

-------------------------------------------------------------------------------

In [ ]:
import os
import torch
import torch.nn as nn
from PIL import Image
import torchvision.transforms as T
import numpy as np
from skimage.metrics import structural_similarity as ssim

# =========================
# 1. 설정
# =========================
DATA_ROOT = "./pix2pix_curated_withMask_1-100/TEST"
TARGET_PATIENT = "100324A"
MODEL_PATH = "./pix2pix_checkpoints/checkpoint_epoch_200.pt"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# 모델 정의
# =========================
class UNetBlock(nn.Module):
    def __init__(self, in_ch, out_ch, submodule=None, innermost=False, outermost=False):
        super().__init__()
        self.outermost = outermost
        downconv = nn.Conv2d(in_ch, out_ch, 4, 2, 1, bias=False)
        downrelu, downnorm = nn.LeakyReLU(0.2, True), nn.BatchNorm2d(out_ch)
        uprelu, upnorm = nn.ReLU(True), nn.BatchNorm2d(in_ch)

        if outermost:
            upconv = nn.ConvTranspose2d(out_ch * 2, in_ch, 4, 2, 1)
            model = [downconv, submodule, uprelu, upconv, nn.Tanh()]
        elif innermost:
            upconv = nn.ConvTranspose2d(out_ch, in_ch, 4, 2, 1, bias=False)
            model = [downrelu, downconv, uprelu, upconv, upnorm]
        else:
            upconv = nn.ConvTranspose2d(out_ch * 2, in_ch, 4, 2, 1, bias=False)
            model = [downrelu, downconv, downnorm, submodule, uprelu, upconv, upnorm]

        self.model = nn.Sequential(*model)

    def forward(self, x):
        return self.model(x) if self.outermost else torch.cat([x, self.model(x)], 1)


class UNetGenerator(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, num_filters=64):
        super().__init__()
        b8 = UNetBlock(num_filters*8, num_filters*8, innermost=True)
        b7 = UNetBlock(num_filters*8, num_filters*8, submodule=b8)
        b6 = UNetBlock(num_filters*8, num_filters*8, submodule=b7)
        b5 = UNetBlock(num_filters*8, num_filters*8, submodule=b6)
        b4 = UNetBlock(num_filters*4, num_filters*8, submodule=b5)
        b3 = UNetBlock(num_filters*2, num_filters*4, submodule=b4)
        b2 = UNetBlock(num_filters, num_filters*2, submodule=b3)
        self.model = UNetBlock(out_ch, num_filters, submodule=b2, outermost=True)

    def forward(self, x):
        return self.model(x)

# =========================
# 모델 로드
# =========================
G = UNetGenerator().to(DEVICE)
ckpt = torch.load(MODEL_PATH, map_location=DEVICE)
G.load_state_dict(ckpt['G_state'] if 'G_state' in ckpt else ckpt)
G.eval()

# =========================
# 전처리
# =========================
tf = T.Compose([
    T.Resize((256, 256)),
    T.ToTensor(),
    T.Normalize((0.5,), (0.5,))
])

rescale = lambda x: (x + 1.0) / 2.0

# =========================
# 데이터 경로
# =========================
patient_base_path = os.path.join(DATA_ROOT, TARGET_PATIENT)
pre_path = os.path.join(patient_base_path, "pre")
post_path = os.path.join(patient_base_path, "post")

# =========================
# SSIM 평가
# =========================
ssim_list = []

if not os.path.exists(pre_path):
    print(f"경로 없음: {pre_path}")
else:
    file_list = sorted(os.listdir(pre_path))

    with torch.no_grad():
        for i, filename in enumerate(file_list):

            img_pre = Image.open(os.path.join(pre_path, filename)).convert("L")
            img_post = Image.open(os.path.join(post_path, filename)).convert("L")

            input_tensor = tf(img_pre).unsqueeze(0).to(DEVICE)
            fake_tensor = G(input_tensor).cpu().squeeze(0)

            # numpy 변환 + 정규화 복원
            fake_np = rescale(fake_tensor[0]).numpy()
            real_np = rescale(tf(img_post).squeeze()).numpy()

            score = ssim(fake_np, real_np, data_range=1.0)
            ssim_list.append(score)

            print(f"[Slice {i}] SSIM: {score:.4f}")

    print("\n===== FINAL RESULT =====")
    print(f"Mean SSIM: {np.mean(ssim_list):.4f}")
    print(f"Std SSIM : {np.std(ssim_list):.4f}")

In [ ]:
-------------------------------------------------------------------------------
마스크 부위 확인
-------------------------------------------------------------------------------

In [ ]:
import os
import torch
import torch.nn as nn
from PIL import Image
import torchvision.transforms as T
import matplotlib.pyplot as plt
import numpy as np

# =========================
# 1. 설정 (CONFIG)
# =========================
# 학습된 모델(.pt)이 저장된 경로와 파일명
MODEL_PATH = "./pix2pix_checkpoints_v3_pattern/pattern_unet_200.pt" 

# 테스트할 환자 데이터 경로 (TEST 또는 TRAIN 폴더 지정)
DATA_ROOT = "./pix2pix_curated/TEST"  
TARGET_PATIENT = "100285A"        # 분석하고 싶은 환자 폴더명

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# 2. 모델 정의 (학습 코드와 구조가 동일해야 함)
# =========================
class UNetBlock(nn.Module):
    def __init__(self, in_ch, out_ch, submodule=None, innermost=False, outermost=False):
        super().__init__()
        self.outermost = outermost
        downconv = nn.Conv2d(in_ch, out_ch, 4, 2, 1, bias=False)
        downrelu = nn.LeakyReLU(0.2, True)
        downnorm = nn.BatchNorm2d(out_ch)
        uprelu = nn.ReLU(True)

        if outermost:
            upconv = nn.ConvTranspose2d(out_ch * 2, in_ch, 4, 2, 1)
            model = [downconv, submodule, uprelu, upconv, nn.Tanh()]
        elif innermost:
            upconv = nn.ConvTranspose2d(out_ch, in_ch, 4, 2, 1, bias=False)
            model = [downrelu, downconv, uprelu, upconv, nn.BatchNorm2d(in_ch)]
        else:
            upconv = nn.ConvTranspose2d(out_ch * 2, in_ch, 4, 2, 1, bias=False)
            model = [downrelu, downconv, downnorm, submodule, uprelu, upconv, nn.BatchNorm2d(in_ch)]
        self.model = nn.Sequential(*model)

    def forward(self, x):
        if self.outermost: return self.model(x)
        return torch.cat([x, self.model(x)], 1)

class UNetGenerator(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, num_filters=64):
        super().__init__()
        b8 = UNetBlock(num_filters*8, num_filters*8, innermost=True)
        b7 = UNetBlock(num_filters*8, num_filters*8, submodule=b8)
        b6 = UNetBlock(num_filters*8, num_filters*8, submodule=b7)
        b5 = UNetBlock(num_filters*8, num_filters*8, submodule=b6)
        b4 = UNetBlock(num_filters*4, num_filters*8, submodule=b5)
        b3 = UNetBlock(num_filters*2, num_filters*4, submodule=b4)
        b2 = UNetBlock(num_filters, num_filters*2, submodule=b3)
        self.model = UNetBlock(out_ch, num_filters, submodule=b2, outermost=True)
    def forward(self, x): return self.model(x)

# =========================
# 3. 모델 로드 및 전처리 설정
# =========================
print(f" Loading Model: {MODEL_PATH}")
G = UNetGenerator(in_ch=1, out_ch=1).to(DEVICE)
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)

# 저장 방식('G' 키 유무)에 따른 로드 처리
if 'G' in checkpoint:
    G.load_state_dict(checkpoint['G'])
else:
    G.load_state_dict(checkpoint)
G.eval()

# 전처리 설정 (Resize & Normalize)
transform = T.Compose([
    T.Resize((256, 256), interpolation=Image.BILINEAR),
    T.ToTensor(),
    T.Normalize((0.5,), (0.5,))
])

# 시각화를 위한 역정규화 함수 (-1~1 -> 0~1)
def rescale(x):
    return (x + 1.0) / 2.0

# =========================
# 4. 이미지 생성 및 패턴 분석 시각화 루틴
# =========================
patient_path = os.path.join(DATA_ROOT, TARGET_PATIENT)
pre_dir = os.path.join(patient_path, "pre")
post_dir = os.path.join(patient_path, "post")

if not os.path.exists(pre_dir) or not os.path.exists(post_dir):
    print(f" 경로 오류: {patient_path} 내부에 pre/post 폴더가 없습니다.")
else:
    file_list = sorted(os.listdir(pre_dir))
    print(f"🔍 Analyzing {TARGET_PATIENT} ({len(file_list)} slices)...")

    # 모든 슬라이스에 대해 루프를 돌며 생성 (plt.show()는 중간에 멈추므로 주의)
    #  Jupyter Notebook이 아닌 환경에서는 plt.savefig()로 바꾸는 것을 권장합니다.
    with torch.no_grad():
        for i, filename in enumerate(file_list):
            # 1. 이미지 로드 및 전처리
            img_pre = Image.open(os.path.join(pre_dir, filename)).convert("L")
            img_post = Image.open(os.path.join(post_path, filename)).convert("L")

            input_tensor = transform(img_pre).unsqueeze(0).to(DEVICE) # Batch 차원 추가

            # 2. 모델 추론 (예측)
            fake_tensor = G(input_tensor).cpu().squeeze(0) # Batch 차원 제거

            # 3. 데이터 변환 (넘파이 배열화 & 역정규화)
            pre_np = rescale(input_tensor.cpu().squeeze()).numpy()
            fake_np = rescale(fake_tensor[0]).numpy()
            
            # 실제 정답(Post)도 비교용으로 변환
            post_tensor = transform(img_post)
            post_np = rescale(post_tensor.squeeze()).numpy()

            # --- 패턴 분석 핵심: Extracted Pattern 계산 ---
            # 모델이 단순 색칠 공부를 했는지, 질감을 파악했는지 확인하기 위해
            # 생성된 Fake 이미지에서 입력 Pre 이미지를 뺍니다. (Residual 추출)
            # 0보다 작은 값(음수)은 버리고 0~1 사이로 클리핑합니다.
            diff_pattern_np = np.clip(fake_np - pre_np, 0, 1)

            # 4. 시각화 (4-패널 비교)
            plt.figure(figsize=(20, 5))
            
            # (1) 입력: 조영 전 패턴 (Pre-contrast)
            plt.subplot(1, 4, 1)
            plt.imshow(pre_np, cmap='gray')
            plt.title(f"Pre (Input Texture)\n{filename}")
            plt.axis('off')

            # (2) 추론: 추출된 증강 패턴 (Extracted Pattern)
            # cmap='hot'을 사용하여 증강된 정도를 온도로 표현 (붉고 노란 곳이 증강 부위)
            plt.subplot(1, 4, 2)
            plt.imshow(diff_pattern_np, cmap='hot') 
            plt.title("Extracted Pattern\n(AI Prediction: Diff)")
            plt.axis('off')

            # (3) 결과: 최종 생성 이미지 (Generated Fake)
            plt.subplot(1, 4, 3)
            plt.imshow(fake_np, cmap='gray')
            plt.title("Generated Fake\n(Final Result)")
            plt.axis('off')

            # (4) 정답: 실제 조영 후 영상 (Ground Truth)
            plt.subplot(1, 4, 4)
            plt.imshow(post_np, cmap='gray')
            plt.title("Real Post\n(Ground Truth)")
            plt.axis('off')

            plt.tight_layout()
            
            # Jupyter Notebook 등에서 루프 돌며 그릴 때plt.show() 사용
            # 만약 슬라이스가 너무 많다면 plt.savefig(f'result_{i}.png')로 저장하는게 좋습니다.
            plt.show() 
            plt.close() # 메모리 해제

print("🏁 패턴 기반 생성 완료.")

======================================================================================================

BASIC_pix2pix에 parameter를 pre, mask 2개로 구성 후 학습

======================================================================================================

In [ ]:

import os

import torch

import torch.nn as nn

import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

from PIL import Image

import torchvision.transforms as T

from tqdm import tqdm



# =========================

# 1. CONFIG (경로 및 하이퍼파라미터)

# =========================

# D드라이브 경로를 사용하시려면 아래 DATA_ROOT를 수정하세요. 

# 예: "D:/BrainData/pix2pix_curated_withMask"

DATA_ROOT = "./pix2pix_curated_withMask" 
CHECKPOINT_DIR = "./pix2pix_checkpoints_v2"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

BATCH_SIZE = 8
EPOCHS = 200
LR_G = 2e-4
LR_D = 2e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LAMBDA_L1 = 100
MASK_WEIGHT = 15.0  # 종양 부위(Mask)에 줄 가중치

print(f"현재 사용 기기: {DEVICE}")

# =========================

# 2. DATASET (Pre + Mask -> 2채널 구성)

# =========================

class Pix2PixMaskDataset(Dataset):
    def __init__(self, root):
        self.samples = []
        self.tf = T.Compose([
            T.Resize((256, 256)),

            T.ToTensor(),

            T.Normalize((0.5,), (0.5,))
        ])

        self.mask_tf = T.Compose([
            T.Resize((256, 256)),
            T.ToTensor() # 마스크는 0~1 사이 값 유지

        ])

        for pid in sorted(os.listdir(root)):
            p_path = os.path.join(root, pid)
            pre_dir, post_dir, mask_dir = os.path.join(p_path, "pre"), os.path.join(p_path, "post"), os.path.join(p_path, "mask")

            if not all(os.path.isdir(d) for d in [pre_dir, post_dir, mask_dir]): continue
                
            for f in sorted(os.listdir(pre_dir)):

                if os.path.exists(os.path.join(post_dir, f)) and os.path.exists(os.path.join(mask_dir, f)):

                    self.samples.append({

                        "pre": os.path.join(pre_dir, f),

                        "post": os.path.join(post_dir, f),

                        "mask": os.path.join(mask_dir, f)

                    })



    def __len__(self): return len(self.samples)



    def __getitem__(self, idx):

        s = self.samples[idx]

        pre_img = Image.open(s["pre"]).convert("L")

        post_img = Image.open(s["post"]).convert("L")

        mask_img = Image.open(s["mask"]).convert("L")



        pre_tensor = self.tf(pre_img)

        post_tensor = self.tf(post_img)

        mask_tensor = self.mask_tf(mask_img)



        # Pre와 Mask를 합쳐서 2채널 입력 생성

        input_tensor = torch.cat([pre_tensor, mask_tensor], dim=0) 

        return input_tensor, post_tensor, mask_tensor



# =========================

# 3. MODEL (U-Net Generator & PatchD)

# =========================

class UNetBlock(nn.Module):

    def __init__(self, in_ch, out_ch, submodule=None, innermost=False, outermost=False):

        super().__init__()

        self.outermost = outermost

        downconv = nn.Conv2d(in_ch, out_ch, 4, 2, 1, bias=False)

        downrelu = nn.LeakyReLU(0.2, True)

        downnorm = nn.BatchNorm2d(out_ch)

        uprelu = nn.ReLU(True)



        if outermost:

            upconv = nn.ConvTranspose2d(out_ch * 2, 1, 4, 2, 1) # 최종 출력 채널 1

            model = [downconv] + [submodule] + [uprelu, upconv, nn.Tanh()]

        elif innermost:

            upconv = nn.ConvTranspose2d(out_ch, in_ch, 4, 2, 1, bias=False)

            model = [downrelu, downconv] + [uprelu, upconv, nn.BatchNorm2d(in_ch)]

        else:

            upconv = nn.ConvTranspose2d(out_ch * 2, in_ch, 4, 2, 1, bias=False)

            model = [downrelu, downconv, downnorm] + [submodule] + [uprelu, upconv, nn.BatchNorm2d(in_ch)]



        self.model = nn.Sequential(*model)



    def forward(self, x):

        if self.outermost: return self.model(x)

        else: return torch.cat([x, self.model(x)], 1)



class UNetGenerator(nn.Module):

    def __init__(self, in_ch=2, num_filters=64):

        super().__init__()

        # 안쪽부터 바깥쪽으로 재귀 구조 생성

        b8 = UNetBlock(num_filters*8, num_filters*8, innermost=True)

        b7 = UNetBlock(num_filters*8, num_filters*8, submodule=b8)

        b6 = UNetBlock(num_filters*8, num_filters*8, submodule=b7)

        b5 = UNetBlock(num_filters*8, num_filters*8, submodule=b6)

        b4 = UNetBlock(num_filters*4, num_filters*8, submodule=b5)

        b3 = UNetBlock(num_filters*2, num_filters*4, submodule=b4)

        b2 = UNetBlock(num_filters, num_filters*2, submodule=b3)

        # [수정됨] 데이터 입력 채널(in_ch=2)을 정확히 전달

        self.model = UNetBlock(in_ch, num_filters, submodule=b2, outermost=True)



    def forward(self, x): return self.model(x)



class PatchDiscriminator(nn.Module):

    def __init__(self, in_ch=3): # (Pre+Mask) 2ch + (Post) 1ch = 3ch

        super().__init__()

        def block(i, o, s=2):

            return nn.Sequential(

                nn.Conv2d(i, o, 4, s, 1, bias=False),

                nn.BatchNorm2d(o),

                nn.LeakyReLU(0.2, inplace=True)

            )

        self.model = nn.Sequential(

            nn.Conv2d(in_ch, 64, 4, 2, 1),

            nn.LeakyReLU(0.2, inplace=True),

            block(64, 128),

            block(128, 256),

            block(256, 512, s=1),

            nn.Conv2d(512, 1, 4, 1, 1)

        )

    def forward(self, x, y): return self.model(torch.cat([x, y], 1))



# =========================

# 4. LOSS FUNCTION & UTILS

# =========================

def mask_weighted_l1_loss(pred, target, mask, weight):

    loss = torch.abs(pred - target)

    # mask=1인 영역엔 weight배, mask=0인 영역엔 1배 가중치 부여

    weight_map = torch.ones_like(mask) + (mask * (weight - 1.0))

    return torch.mean(loss * weight_map)



# =========================

# 5. TRAINING LOOP

# =========================

train_loader = DataLoader(Pix2PixMaskDataset(f"{DATA_ROOT}/TRAIN"), batch_size=BATCH_SIZE, shuffle=True)



G = UNetGenerator(in_ch=2).to(DEVICE)

D = PatchDiscriminator(in_ch=3).to(DEVICE)



opt_G = optim.Adam(G.parameters(), lr=LR_G, betas=(0.5, 0.999))

opt_D = optim.Adam(D.parameters(), lr=LR_D, betas=(0.5, 0.999))

criterion_GAN = nn.BCEWithLogitsLoss()



for epoch in range(EPOCHS):

    G.train(); D.train()

    loop = tqdm(train_loader, leave=False)

    for inputs, post, mask in loop:

        inputs, post, mask = inputs.to(DEVICE), post.to(DEVICE), mask.to(DEVICE)



        # Discriminator Update

        fake = G(inputs)

        real_pred = D(inputs, post)

        fake_pred = D(inputs, fake.detach())

        loss_D = (criterion_GAN(real_pred, torch.ones_like(real_pred)) + 

                  criterion_GAN(fake_pred, torch.zeros_like(fake_pred))) * 0.5

        opt_D.zero_grad(); loss_D.backward(); opt_D.step()



        # Generator Update

        fake_pred = D(inputs, fake)

        loss_G_GAN = criterion_GAN(fake_pred, torch.ones_like(fake_pred))

        loss_G_L1 = mask_weighted_l1_loss(fake, post, mask, MASK_WEIGHT) * LAMBDA_L1

        loss_G = loss_G_GAN + loss_G_L1

        opt_G.zero_grad(); loss_G.backward(); opt_G.step()



        loop.set_description(f"Epoch [{epoch+1}/{EPOCHS}]")

        loop.set_postfix(G_L1=loss_G_L1.item(), D_loss=loss_D.item())



    # 체크포인트 저장 (50 에포크마다)

    if (epoch + 1) % 50 == 0:

        torch.save(G.state_dict(), f"{CHECKPOINT_DIR}/G_epoch_{epoch+1}.pt")

        print(f"✔️ Epoch {epoch+1} 저장 완료!")



print(" 모든 학습이 완료되었습니다!")

-------------------------------------------------------------------------------

모델 생성 테스트

-------------------------------------------------------------------------------

In [ ]:
import os
import torch
import torch.nn as nn
from PIL import Image
import torchvision.transforms as T
import matplotlib.pyplot as plt

# =========================
# 1. 설정 (CONFIG)
# =========================
DATA_ROOT = "./pix2pix_curated_withMask/TEST" # TEST 데이터 경로
TARGET_PATIENT = "100126B"                   # 분석할 환자 ID
MODEL_PATH = "./pix2pix_checkpoints_v2/G_epoch_200.pt" # 저장된 체크포인트 경로
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# 2. 모델 정의 (2-Channel UNet)
# =========================
class UNetBlock(nn.Module):
    def __init__(self, in_ch, out_ch, submodule=None, innermost=False, outermost=False):
        super().__init__()
        self.outermost = outermost
        downconv = nn.Conv2d(in_ch, out_ch, 4, 2, 1, bias=False)
        downrelu = nn.LeakyReLU(0.2, True)
        downnorm = nn.BatchNorm2d(out_ch)
        uprelu = nn.ReLU(True)

        if outermost:
            upconv = nn.ConvTranspose2d(out_ch * 2, 1, 4, 2, 1) # 최종 출력 1채널
            model = [downconv] + [submodule] + [uprelu, upconv, nn.Tanh()]
        elif innermost:
            upconv = nn.ConvTranspose2d(out_ch, in_ch, 4, 2, 1, bias=False)
            model = [downrelu, downconv] + [uprelu, upconv, nn.BatchNorm2d(in_ch)]
        else:
            upconv = nn.ConvTranspose2d(out_ch * 2, in_ch, 4, 2, 1, bias=False)
            model = [downrelu, downconv, downnorm] + [submodule] + [uprelu, upconv, nn.BatchNorm2d(in_ch)]

        self.model = nn.Sequential(*model)

    def forward(self, x):
        if self.outermost: return self.model(x)
        else: return torch.cat([x, self.model(x)], 1)

class UNetGenerator(nn.Module):
    def __init__(self, in_ch=2, num_filters=64):
        super().__init__()
        b8 = UNetBlock(num_filters*8, num_filters*8, innermost=True)
        b7 = UNetBlock(num_filters*8, num_filters*8, submodule=b8)
        b6 = UNetBlock(num_filters*8, num_filters*8, submodule=b7)
        b5 = UNetBlock(num_filters*8, num_filters*8, submodule=b6)
        b4 = UNetBlock(num_filters*4, num_filters*8, submodule=b5)
        b3 = UNetBlock(num_filters*2, num_filters*4, submodule=b4)
        b2 = UNetBlock(num_filters, num_filters*2, submodule=b3)
        # 입력 채널 in_ch=2 (Pre + Mask)
        self.model = UNetBlock(in_ch, num_filters, submodule=b2, outermost=True)

    def forward(self, x): return self.model(x)

# =========================
# 3. 모델 로드 및 실행
# =========================
# 모델 초기화 (입력 2채널)
G = UNetGenerator(in_ch=2).to(DEVICE)

# 가중치 로드
if os.path.exists(MODEL_PATH):
    ckpt = torch.load(MODEL_PATH, map_location=DEVICE)
    # 가중치 딕셔너리 구조에 따라 대응 (G_state 키가 있을 경우와 없을 경우 모두 처리)
    G.load_state_dict(ckpt['G_state'] if isinstance(ckpt, dict) and 'G_state' in ckpt else ckpt)
    G.eval()
    print(f" 모델 로드 성공: {MODEL_PATH}")
else:
    print(f" 모델 파일을 찾을 수 없습니다: {MODEL_PATH}")

# 전처리 및 후처리(Rescale) 설정
tf = T.Compose([T.Resize((256, 256)), T.ToTensor(), T.Normalize((0.5,), (0.5,))])
mask_tf = T.Compose([T.Resize((256, 256)), T.ToTensor()])
rescale = lambda x: (x + 1.0) / 2.0

# 환자별 폴더 경로
patient_path = os.path.join(DATA_ROOT, TARGET_PATIENT)
pre_dir, mask_dir, post_dir = [os.path.join(patient_path, d) for d in ["pre", "mask", "post"]]

if not os.path.exists(pre_dir):
    print(f" '{TARGET_PATIENT}' 환자의 데이터를 찾을 수 없습니다.")
else:
    file_list = sorted(os.listdir(pre_dir))
    print(f" 분석 시작: {TARGET_PATIENT} (총 {len(file_list)} 슬라이스)")

    with torch.no_grad():
        for filename in file_list:
            # 1. 이미지 로드 및 흑백 변환
            img_pre = Image.open(os.path.join(pre_dir, filename)).convert("L")
            img_mask = Image.open(os.path.join(mask_dir, filename)).convert("L")
            img_post = Image.open(os.path.join(post_dir, filename)).convert("L")

            # 2. 전처리 및 2채널 결합
            pre_t = tf(img_pre)
            mask_t = mask_tf(img_mask)
            # [1, 2, 256, 256] 형태의 입력 텐서 생성
            input_tensor = torch.cat([pre_t, mask_t], dim=0).unsqueeze(0).to(DEVICE)

            # 3. 모델 추론
            fake_tensor = G(input_tensor).cpu().squeeze(0)

            # 4. 결과 시각화
            plt.figure(figsize=(18, 5))
            
            imgs = [rescale(pre_t.squeeze()), mask_t.squeeze(), rescale(fake_tensor.squeeze()), rescale(tf(img_post).squeeze())]
            titles = [f"Input: Pre ({filename})", "Input: Mask", "Generated (Fake)", "Real (Target)"]
            
            for i, (img, title) in enumerate(zip(imgs, titles)):
                plt.subplot(1, 4, i + 1)
                plt.imshow(img, cmap='gray')
                plt.title(title)
                plt.axis('off')

            plt.tight_layout()
            plt.show()

print(" 모든 슬라이스 분석 완료")

==========================================================================================

MASK GENERATE Model Training

==========================================================================================

In [ ]:
import os
import glob
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
from tqdm import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TRAIN_DIR = "./pix2pix_curated_withMask/TRAIN"
MODEL_SAVE = "./tumor_mask_unet.pth"

IMG_SIZE = 256
BATCH = 8
EPOCHS = 50  # 100 -> 50으로 수정
LR = 1e-4

# -------------------------
# Dataset
# -------------------------
class TumorDataset(Dataset):
    def __init__(self, root):
        self.pre = []
        self.mask = []
        patients = sorted(os.listdir(root))
        for p in patients:
            pre_dir = os.path.join(root, p, "pre")
            mask_dir = os.path.join(root, p, "mask")
            pre_list = sorted(glob.glob(pre_dir + "/*"))
            mask_list = sorted(glob.glob(mask_dir + "/*"))
            for a, b in zip(pre_list, mask_list):
                self.pre.append(a)
                self.mask.append(b)

        self.tf = T.Compose([
            T.Resize((IMG_SIZE, IMG_SIZE)),
            T.ToTensor()
        ])

    def __len__(self):
        return len(self.pre)

    def __getitem__(self, idx):
        pre = Image.open(self.pre[idx]).convert("L")
        mask = Image.open(self.mask[idx]).convert("L")
        pre = self.tf(pre)
        mask = self.tf(mask)
        mask = (mask > 0.5).float()
        return pre, mask

# -------------------------
# UNet
# -------------------------
def CBR(a,b):
    return nn.Sequential(
        nn.Conv2d(a,b,3,1,1),
        nn.BatchNorm2d(b),
        nn.ReLU()
    )

class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.d1 = nn.Sequential(CBR(1,64),CBR(64,64))
        self.p1 = nn.MaxPool2d(2)
        self.d2 = nn.Sequential(CBR(64,128),CBR(128,128))
        self.p2 = nn.MaxPool2d(2)
        self.d3 = nn.Sequential(CBR(128,256),CBR(256,256))
        self.p3 = nn.MaxPool2d(2)
        self.b = nn.Sequential(CBR(256,512),CBR(512,512))
        self.u3 = nn.ConvTranspose2d(512,256,2,2)
        self.c3 = nn.Sequential(CBR(512,256),CBR(256,256))
        self.u2 = nn.ConvTranspose2d(256,128,2,2)
        self.c2 = nn.Sequential(CBR(256,128),CBR(128,128))
        self.u1 = nn.ConvTranspose2d(128,64,2,2)
        self.c1 = nn.Sequential(CBR(128,64),CBR(64,64))
        self.out = nn.Conv2d(64,1,1)

    def forward(self,x):
        d1 = self.d1(x)
        d2 = self.d2(self.p1(d1))
        d3 = self.d3(self.p2(d2))
        b = self.b(self.p3(d3))
        u3 = self.u3(b)
        u3 = torch.cat([u3,d3],1)
        u3 = self.c3(u3)
        u2 = self.u2(u3)
        u2 = torch.cat([u2,d2],1)
        u2 = self.c2(u2)
        u1 = self.u1(u2)
        u1 = torch.cat([u1,d1],1)
        u1 = self.c1(u1)
        return torch.sigmoid(self.out(u1))

# -------------------------
# Dice Loss
# -------------------------
def dice(pred, target):
    smooth = 1e-5
    p = pred.view(-1)
    t = target.view(-1)
    inter = (p*t).sum()
    return 1 - (2*inter+smooth)/(p.sum()+t.sum()+smooth)

# -------------------------
# Train
# -------------------------
dataset = TumorDataset(TRAIN_DIR)
loader = DataLoader(dataset,batch_size=BATCH,shuffle=True)

model = UNet().to(DEVICE)
opt = torch.optim.Adam(model.parameters(),lr=LR)
bce = nn.BCELoss()

print(f"Starting training for {EPOCHS} epochs...")

for e in range(1, EPOCHS + 1): # 1부터 50까지

    model.train()
    bar = tqdm(loader)
    epoch_loss = 0

    for pre, mask in bar:
        pre = pre.to(DEVICE)
        mask = mask.to(DEVICE)

        pred = model(pre)
        loss = bce(pred,mask) + dice(pred,mask)

        opt.zero_grad()
        loss.backward()
        opt.step()

        epoch_loss += loss.item()
        bar.set_description(f"epoch {e}/{EPOCHS} loss {loss.item():.4f}")

    # 10에포크마다 체크포인트 저장
    if e % 10 == 0:
        checkpoint_path = f"tumor_mask_unet_epoch_{e}.pth"
        torch.save(model.state_dict(), checkpoint_path)
        print(f"\nCheckpoint saved: {checkpoint_path}")

# 최종 모델 저장
torch.save(model.state_dict(), MODEL_SAVE)
print(f"Final model saved: {MODEL_SAVE}")
print("training finished")

-------------------------------------------------------------------------------

++이이서 학습++

-------------------------------------------------------------------------------

In [ ]:
import os
import glob
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
from tqdm import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TRAIN_DIR = "./pix2pix_curated_withMask/TRAIN"

LOAD_MODEL = "./mask_generate_model/tumor_mask_unet_epoch100.pth"   # 기존 모델
MODEL_SAVE = "./mask_generate_model/tumor_mask_unet_epoch200.pth"

IMG_SIZE = 256
BATCH = 8
EPOCHS = 100     # 추가 학습
START_EPOCH = 101
LR = 1e-4

# -------------------------
# Dataset
# -------------------------
class TumorDataset(Dataset):
    def __init__(self, root):

        self.pre = []
        self.mask = []

        patients = sorted(os.listdir(root))

        for p in patients:

            pre_dir = os.path.join(root, p, "pre")
            mask_dir = os.path.join(root, p, "mask")

            pre_list = sorted(glob.glob(pre_dir + "/*"))
            mask_list = sorted(glob.glob(mask_dir + "/*"))

            for a, b in zip(pre_list, mask_list):
                self.pre.append(a)
                self.mask.append(b)

        self.tf = T.Compose([
            T.Resize((IMG_SIZE, IMG_SIZE)),
            T.ToTensor()
        ])

    def __len__(self):
        return len(self.pre)

    def __getitem__(self, idx):

        pre = Image.open(self.pre[idx]).convert("L")
        mask = Image.open(self.mask[idx]).convert("L")

        pre = self.tf(pre)
        mask = self.tf(mask)

        mask = (mask > 0.5).float()

        return pre, mask


# -------------------------
# UNet
# -------------------------
def CBR(a,b):
    return nn.Sequential(
        nn.Conv2d(a,b,3,1,1),
        nn.BatchNorm2d(b),
        nn.ReLU()
    )

class UNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.d1 = nn.Sequential(CBR(1,64),CBR(64,64))
        self.p1 = nn.MaxPool2d(2)

        self.d2 = nn.Sequential(CBR(64,128),CBR(128,128))
        self.p2 = nn.MaxPool2d(2)

        self.d3 = nn.Sequential(CBR(128,256),CBR(256,256))
        self.p3 = nn.MaxPool2d(2)

        self.b = nn.Sequential(CBR(256,512),CBR(512,512))

        self.u3 = nn.ConvTranspose2d(512,256,2,2)
        self.c3 = nn.Sequential(CBR(512,256),CBR(256,256))

        self.u2 = nn.ConvTranspose2d(256,128,2,2)
        self.c2 = nn.Sequential(CBR(256,128),CBR(128,128))

        self.u1 = nn.ConvTranspose2d(128,64,2,2)
        self.c1 = nn.Sequential(CBR(128,64),CBR(64,64))

        self.out = nn.Conv2d(64,1,1)

    def forward(self,x):

        d1 = self.d1(x)
        d2 = self.d2(self.p1(d1))
        d3 = self.d3(self.p2(d2))

        b = self.b(self.p3(d3))

        u3 = self.u3(b)
        u3 = torch.cat([u3,d3],1)
        u3 = self.c3(u3)

        u2 = self.u2(u3)
        u2 = torch.cat([u2,d2],1)
        u2 = self.c2(u2)

        u1 = self.u1(u2)
        u1 = torch.cat([u1,d1],1)
        u1 = self.c1(u1)

        return torch.sigmoid(self.out(u1))


# -------------------------
# Dice Loss
# -------------------------
def dice(pred, target):

    smooth = 1e-5

    p = pred.view(-1)
    t = target.view(-1)

    inter = (p*t).sum()

    return 1 - (2*inter+smooth)/(p.sum()+t.sum()+smooth)


# -------------------------
# Data
# -------------------------
dataset = TumorDataset(TRAIN_DIR)
loader = DataLoader(dataset,batch_size=BATCH,shuffle=True)

# -------------------------
# Model
# -------------------------
model = UNet().to(DEVICE)

print("loading pretrained model...")
model.load_state_dict(torch.load(LOAD_MODEL))

opt = torch.optim.Adam(model.parameters(),lr=LR)
bce = nn.BCELoss()

print("continue training...")

# -------------------------
# Train
# -------------------------
for e in range(START_EPOCH, START_EPOCH + EPOCHS):

    model.train()

    bar = tqdm(loader)

    for pre, mask in bar:

        pre = pre.to(DEVICE)
        mask = mask.to(DEVICE)

        pred = model(pre)

        loss = bce(pred,mask) + dice(pred,mask)

        opt.zero_grad()
        loss.backward()
        opt.step()

        bar.set_description(f"epoch {e} loss {loss.item():.4f}")

    if e % 10 == 0:

        ckpt = f"tumor_mask_unet_epoch_{e}.pth"
        torch.save(model.state_dict(), ckpt)

        print("saved:", ckpt)


torch.save(model.state_dict(), MODEL_SAVE)

print("training finished")

-------------------------------------------------------------------------------

모델 생성 테스트

-------------------------------------------------------------------------------

In [ ]:
import os
import glob
import torch
import torch.nn as nn
from PIL import Image
import torchvision.transforms as T
import numpy as np
import matplotlib.pyplot as plt

# ==========================================
# 1. 설정 (여기서 환자 폴더 이름을 지정하세요)
# ==========================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TEST_DIR = "./pix2pix_curated_withMask/TEST"
MODEL_PATH = "./mask_generate_model/tumor_mask_unet_epoch100.pth"
IMG_SIZE = 256

# 보고 싶은 특정 환자 폴더명 입력
TARGET_PATIENT = "100380B" 
# ==========================================

# -----------------
# same UNet
# -----------------
def CBR(a,b):
    return nn.Sequential(
        nn.Conv2d(a,b,3,1,1),
        nn.BatchNorm2d(b),
        nn.ReLU()
    )

class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.d1 = nn.Sequential(CBR(1,64),CBR(64,64))
        self.p1 = nn.MaxPool2d(2)
        self.d2 = nn.Sequential(CBR(64,128),CBR(128,128))
        self.p2 = nn.MaxPool2d(2)
        self.d3 = nn.Sequential(CBR(128,256),CBR(256,256))
        self.p3 = nn.MaxPool2d(2)
        self.b = nn.Sequential(CBR(256,512),CBR(512,512))
        self.u3 = nn.ConvTranspose2d(512,256,2,2)
        self.c3 = nn.Sequential(CBR(512,256),CBR(256,256))
        self.u2 = nn.ConvTranspose2d(256,128,2,2)
        self.c2 = nn.Sequential(CBR(256,128),CBR(128,128))
        self.u1 = nn.ConvTranspose2d(128,64,2,2)
        self.c1 = nn.Sequential(CBR(128,64),CBR(64,64))
        self.out = nn.Conv2d(64,1,1)

    def forward(self,x):
        d1 = self.d1(x)
        d2 = self.d2(self.p1(d1))
        d3 = self.d3(self.p2(d2))
        b = self.b(self.p3(d3))
        u3 = self.u3(b)
        u3 = torch.cat([u3,d3],1)
        u3 = self.c3(u3)
        u2 = self.u2(u3)
        u2 = torch.cat([u2,d2],1)
        u2 = self.c2(u2)
        u1 = self.u1(u2)
        u1 = torch.cat([u1,d1],1)
        u1 = self.c1(u1)
        return torch.sigmoid(self.out(u1))

# 모델 로드
model = UNet().to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

tf = T.Compose([
    T.Resize((IMG_SIZE,IMG_SIZE)),
    T.ToTensor()
])

# 시각화 실행 (특정 환자만)
pre_dir = os.path.join(TEST_DIR, TARGET_PATIENT, "pre")
post_dir = os.path.join(TEST_DIR, TARGET_PATIENT, "post")
mask_dir = os.path.join(TEST_DIR, TARGET_PATIENT, "mask")

if not os.path.exists(pre_dir):
    print(f"Error: 폴더를 찾을 수 없습니다 -> {pre_dir}")
else:
    files = sorted(glob.glob(pre_dir + "/*"))
    n = len(files)

    if n > 0:
        with torch.no_grad():
            # (슬라이스 수, 4개 컬럼) 격자 생성
            fig, axes = plt.subplots(n, 4, figsize=(12, 3 * n))
            if n == 1: axes = np.expand_dims(axes, axis=0)
            
            plt.suptitle(f"Target Patient: {TARGET_PATIENT}", fontsize=15)

            for i, f in enumerate(files):
                name = os.path.basename(f)

                # 1. PRE (Input)
                pre_img = Image.open(f).convert("L")
                pre_np = np.array(pre_img.resize((IMG_SIZE,IMG_SIZE)))
                pre_tensor = tf(pre_img).unsqueeze(0).to(DEVICE)

                # 2. PREDICT (Model Output)
                pred = model(pre_tensor)
                pred = (pred>0.5).float().squeeze().cpu().numpy()

                # 3. POST (Actual Target)
                post_path = os.path.join(post_dir, name)
                post_np = np.array(Image.open(post_path).convert("L").resize((IMG_SIZE,IMG_SIZE))) if os.path.exists(post_path) else np.zeros((IMG_SIZE,IMG_SIZE))

                # 4. GT (Answer Mask)
                gt_path = os.path.join(mask_dir, name)
                gt_np = np.array(Image.open(gt_path).convert("L").resize((IMG_SIZE,IMG_SIZE))) if os.path.exists(gt_path) else np.zeros((IMG_SIZE,IMG_SIZE))

                # 한 줄에 4개 이미지 배치
                imgs = [pre_np, post_np, gt_np, pred]
                titles = ["PRE", "POST", "GT MASK", "PRED MASK"]
                
                for j in range(4):
                    axes[i, j].imshow(imgs[j], cmap="gray")
                    if i == 0: axes[i, j].set_title(titles[j])
                    axes[i, j].axis("off")
                
                # 왼쪽에 슬라이스 파일명 표시
                axes[i, 0].text(-20, IMG_SIZE//2, name, rotation=90, va='center', ha='right', fontsize=8)

            plt.tight_layout()
            plt.show()
    else:
        print("슬라이스 파일이 없습니다.")

=======================================================================================

pre->fake_mask로 나온 데이터셋을 곧바로 pre,mask->post모델에 적용하여 생성 테스트

=======================================================================================

In [ ]:
import os
import torch
import torch.nn as nn
from PIL import Image
import torchvision.transforms as T
import matplotlib.pyplot as plt
import numpy as np

# ======================================================
# CONFIG
# ======================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TEST_DIR = "./pix2pix_curated_withMask/TEST"
TARGET_PATIENT = "100380B"

MASK_MODEL_PATH = "./mask_generate_model/tumor_mask_unet_epoch100.pth"
POST_MODEL_PATH = "./pix2pix_checkpoints_v2/G_epoch_200.pt"

IMG_SIZE = 256


# ======================================================
# MASK UNET
# ======================================================

def CBR(a,b):
    return nn.Sequential(
        nn.Conv2d(a,b,3,1,1),
        nn.BatchNorm2d(b),
        nn.ReLU()
    )

class MaskUNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.d1 = nn.Sequential(CBR(1,64),CBR(64,64))
        self.p1 = nn.MaxPool2d(2)

        self.d2 = nn.Sequential(CBR(64,128),CBR(128,128))
        self.p2 = nn.MaxPool2d(2)

        self.d3 = nn.Sequential(CBR(128,256),CBR(256,256))
        self.p3 = nn.MaxPool2d(2)

        self.b = nn.Sequential(CBR(256,512),CBR(512,512))

        self.u3 = nn.ConvTranspose2d(512,256,2,2)
        self.c3 = nn.Sequential(CBR(512,256),CBR(256,256))

        self.u2 = nn.ConvTranspose2d(256,128,2,2)
        self.c2 = nn.Sequential(CBR(256,128),CBR(128,128))

        self.u1 = nn.ConvTranspose2d(128,64,2,2)
        self.c1 = nn.Sequential(CBR(128,64),CBR(64,64))

        self.out = nn.Conv2d(64,1,1)

    def forward(self,x):

        d1 = self.d1(x)
        d2 = self.d2(self.p1(d1))
        d3 = self.d3(self.p2(d2))

        b = self.b(self.p3(d3))

        u3 = self.u3(b)
        u3 = torch.cat([u3,d3],1)
        u3 = self.c3(u3)

        u2 = self.u2(u3)
        u2 = torch.cat([u2,d2],1)
        u2 = self.c2(u2)

        u1 = self.u1(u2)
        u1 = torch.cat([u1,d1],1)
        u1 = self.c1(u1)

        return torch.sigmoid(self.out(u1))


# ======================================================
# PIX2PIX GENERATOR
# ======================================================

class UNetBlock(nn.Module):

    def __init__(self,in_ch,out_ch,submodule=None,innermost=False,outermost=False):

        super().__init__()

        self.outermost = outermost

        downconv = nn.Conv2d(in_ch,out_ch,4,2,1,bias=False)
        downrelu = nn.LeakyReLU(0.2,True)
        downnorm = nn.BatchNorm2d(out_ch)

        uprelu = nn.ReLU(True)

        if outermost:

            upconv = nn.ConvTranspose2d(out_ch*2,1,4,2,1)

            model = [downconv] + [submodule] + [uprelu,upconv,nn.Tanh()]

        elif innermost:

            upconv = nn.ConvTranspose2d(out_ch,in_ch,4,2,1,bias=False)

            model = [downrelu,downconv] + [uprelu,upconv,nn.BatchNorm2d(in_ch)]

        else:

            upconv = nn.ConvTranspose2d(out_ch*2,in_ch,4,2,1,bias=False)

            model = [downrelu,downconv,downnorm] + [submodule] + [uprelu,upconv,nn.BatchNorm2d(in_ch)]

        self.model = nn.Sequential(*model)

    def forward(self,x):

        if self.outermost:
            return self.model(x)

        else:
            return torch.cat([x,self.model(x)],1)


class UNetGenerator(nn.Module):

    def __init__(self,in_ch=2,num_filters=64):

        super().__init__()

        b8 = UNetBlock(num_filters*8,num_filters*8,innermost=True)
        b7 = UNetBlock(num_filters*8,num_filters*8,submodule=b8)
        b6 = UNetBlock(num_filters*8,num_filters*8,submodule=b7)
        b5 = UNetBlock(num_filters*8,num_filters*8,submodule=b6)

        b4 = UNetBlock(num_filters*4,num_filters*8,submodule=b5)
        b3 = UNetBlock(num_filters*2,num_filters*4,submodule=b4)
        b2 = UNetBlock(num_filters,num_filters*2,submodule=b3)

        self.model = UNetBlock(in_ch,num_filters,submodule=b2,outermost=True)

    def forward(self,x):
        return self.model(x)


# ======================================================
# 모델 로드
# ======================================================

mask_model = MaskUNet().to(DEVICE)
mask_model.load_state_dict(torch.load(MASK_MODEL_PATH,map_location=DEVICE, weights_only=True))
mask_model.eval()

G = UNetGenerator(in_ch=2).to(DEVICE)

ckpt = torch.load(POST_MODEL_PATH,map_location=DEVICE, weights_only=True)
G.load_state_dict(ckpt['G_state'] if isinstance(ckpt,dict) and 'G_state' in ckpt else ckpt)

G.eval()


# ======================================================
# 경로
# ======================================================

patient_dir = os.path.join(TEST_DIR,TARGET_PATIENT)

pre_dir  = os.path.join(patient_dir,"pre")
mask_dir = os.path.join(patient_dir,"mask")   # real mask (visualization only)
post_dir = os.path.join(patient_dir,"post")   # real post (visualization only)

files = sorted(os.listdir(pre_dir))


# ======================================================
# 전처리
# ======================================================

tf_pre = T.Compose([
    T.Resize((IMG_SIZE,IMG_SIZE)),
    T.ToTensor()
])

tf_post = T.Compose([
    T.Resize((IMG_SIZE,IMG_SIZE)),
    T.ToTensor(),
    T.Normalize((0.5,),(0.5,))
])

rescale = lambda x:(x+1)/2


# ======================================================
# 추론
# ======================================================

fig,axes = plt.subplots(len(files),5,figsize=(15,3*len(files)))

if len(files)==1:
    axes = np.expand_dims(axes,0)

with torch.no_grad():

    for i,name in enumerate(files):

        pre_img = Image.open(os.path.join(pre_dir,name)).convert("L")
        real_mask = Image.open(os.path.join(mask_dir,name)).convert("L")
        real_post = Image.open(os.path.join(post_dir,name)).convert("L")

        pre_tensor = tf_pre(pre_img).unsqueeze(0).to(DEVICE)

        # --------------------------------
        # 1. MASK 생성
        # --------------------------------

        fake_mask = mask_model(pre_tensor)
        fake_mask = (fake_mask>0.5).float()

        # --------------------------------
        # 2. POST 생성
        # --------------------------------

        pre_pix = tf_post(pre_img)
        fake_mask_pix = fake_mask.squeeze(0).cpu()

        input_tensor = torch.cat([pre_pix,fake_mask_pix],0).unsqueeze(0).to(DEVICE)

        fake_post = G(input_tensor).cpu()

        # --------------------------------
        # numpy 변환
        # --------------------------------

        pre_np = np.array(pre_img.resize((IMG_SIZE,IMG_SIZE)))
        real_mask_np = np.array(real_mask.resize((IMG_SIZE,IMG_SIZE)))
        fake_mask_np = fake_mask.squeeze().cpu().numpy()

        fake_post_np = rescale(fake_post.squeeze()).numpy()
        real_post_np = np.array(real_post.resize((IMG_SIZE,IMG_SIZE)))/255

        imgs = [pre_np,real_mask_np,fake_mask_np,fake_post_np,real_post_np]

        titles = ["PRE","REAL_MASK","FAKE_MASK","FAKE_POST","REAL_POST"]

        for j in range(5):

            axes[i,j].imshow(imgs[j],cmap="gray")

            if i==0:
                axes[i,j].set_title(titles[j])

            axes[i,j].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
두 모델을 하나의 모델로 합치는 시도_v1

단일 UNET구조

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
from tqdm import tqdm
import cv2
import numpy as np

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATA_ROOT = "./pix2pix_curated_withMask"
CHECKPOINT_DIR = "./joint_model_ckp2"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

BATCH = 8
EPOCHS = 200

LR_MASK = 1e-4
LR_G = 2e-4
LR_D = 2e-4

LAMBDA_L1 = 100
MASK_WEIGHT = 15.0
MASK_LOSS_WEIGHT = 1.0

FREEZE_MASK = False


# =========================
# CLAHE 함수
# =========================

def apply_clahe(img):

    img = np.array(img)

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8,8)
    )

    img = clahe.apply(img)

    return Image.fromarray(img)


# =========================
# Dataset
# =========================

class JointDataset(Dataset):

    def __init__(self, root):

        self.samples = []

        self.tf = T.Compose([
            T.Resize((256,256)),
            T.ToTensor(),
            T.Normalize((0.5,), (0.5,))
        ])

        self.mask_tf = T.Compose([
            T.Resize((256,256)),
            T.ToTensor()
        ])

        for pid in sorted(os.listdir(root)):
            p_path = os.path.join(root, pid)

            pre_dir = os.path.join(p_path,"pre")
            post_dir = os.path.join(p_path,"post")
            mask_dir = os.path.join(p_path,"mask")

            if not os.path.isdir(pre_dir):
                continue

            for f in sorted(os.listdir(pre_dir)):

                pre = os.path.join(pre_dir,f)
                post = os.path.join(post_dir,f)
                mask = os.path.join(mask_dir,f)

                if os.path.exists(post) and os.path.exists(mask):

                    self.samples.append((pre,post,mask))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self,idx):

        pre,post,mask = self.samples[idx]

        pre = Image.open(pre).convert("L")
        post = Image.open(post).convert("L")
        mask = Image.open(mask).convert("L")

        pre = apply_clahe(pre)
        post = apply_clahe(post)

        pre = self.tf(pre)
        post = self.tf(post)
        mask = self.mask_tf(mask)

        mask = (mask>0.5).float()

        return pre,post,mask


# =========================
# Mask Network
# =========================

def CBR(a,b):
    return nn.Sequential(
        nn.Conv2d(a,b,3,1,1),
        nn.BatchNorm2d(b),
        nn.ReLU()
    )

class MaskUNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.d1 = nn.Sequential(CBR(1,64),CBR(64,64))
        self.p1 = nn.MaxPool2d(2)

        self.d2 = nn.Sequential(CBR(64,128),CBR(128,128))
        self.p2 = nn.MaxPool2d(2)

        self.d3 = nn.Sequential(CBR(128,256),CBR(256,256))
        self.p3 = nn.MaxPool2d(2)

        self.b = nn.Sequential(CBR(256,512),CBR(512,512))

        self.u3 = nn.ConvTranspose2d(512,256,2,2)
        self.c3 = nn.Sequential(CBR(512,256),CBR(256,256))

        self.u2 = nn.ConvTranspose2d(256,128,2,2)
        self.c2 = nn.Sequential(CBR(256,128),CBR(128,128))

        self.u1 = nn.ConvTranspose2d(128,64,2,2)
        self.c1 = nn.Sequential(CBR(128,64),CBR(64,64))

        self.out = nn.Conv2d(64,1,1)

    def forward(self,x):

        d1 = self.d1(x)
        d2 = self.d2(self.p1(d1))
        d3 = self.d3(self.p2(d2))

        b = self.b(self.p3(d3))

        u3 = self.u3(b)
        u3 = torch.cat([u3,d3],1)
        u3 = self.c3(u3)

        u2 = self.u2(u3)
        u2 = torch.cat([u2,d2],1)
        u2 = self.c2(u2)

        u1 = self.u1(u2)
        u1 = torch.cat([u1,d1],1)
        u1 = self.c1(u1)

        return torch.sigmoid(self.out(u1))


# =========================
# Generator
# =========================

class Generator(nn.Module):

    def __init__(self):

        super().__init__()

        self.model = nn.Sequential(

            nn.Conv2d(2,64,4,2,1),
            nn.LeakyReLU(0.2),

            nn.Conv2d(64,128,4,2,1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),

            nn.ConvTranspose2d(128,64,4,2,1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.ConvTranspose2d(64,1,4,2,1),
            nn.Tanh()
        )

    def forward(self,x):
        return self.model(x)


# =========================
# Discriminator
# =========================

class PatchD(nn.Module):

    def __init__(self):

        super().__init__()

        self.model = nn.Sequential(

            nn.Conv2d(3,64,4,2,1),
            nn.LeakyReLU(0.2),

            nn.Conv2d(64,128,4,2,1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),

            nn.Conv2d(128,1,4,1,1)
        )

    def forward(self,x,y):

        inp = torch.cat([x,y],1)
        return self.model(inp)


# =========================
# Joint Model
# =========================

class JointModel(nn.Module):

    def __init__(self):

        super().__init__()

        self.mask_net = MaskUNet()
        self.generator = Generator()

    def forward(self, pre, mask_gt=None):

        pred_mask = self.mask_net(pre)

        if self.training:
            mask_for_gen = mask_gt
        else:
            mask_for_gen = pred_mask

        gen_input = torch.cat([pre,mask_for_gen],1)

        fake_post = self.generator(gen_input)

        return pred_mask, fake_post


# =========================
# Loss
# =========================

def dice(pred,target):

    smooth=1e-5

    p=pred.view(-1)
    t=target.view(-1)

    inter=(p*t).sum()

    return 1-(2*inter+smooth)/(p.sum()+t.sum()+smooth)


def mask_weighted_l1(pred,target,mask,weight):

    loss=torch.abs(pred-target)

    weight_map=torch.ones_like(mask)+(mask*(weight-1))

    return torch.mean(loss*weight_map)


# =========================
# Train
# =========================

print(f"[진행] 사용 중인 디바이스: {DEVICE}")
print(f"[진행] 데이터셋 로딩 중: {DATA_ROOT}/TRAIN")

dataset = JointDataset(f"{DATA_ROOT}/TRAIN")
loader = DataLoader(dataset,batch_size=BATCH,shuffle=True)

print(f"[진행] 데이터셋 로딩 완료. 총 {len(dataset)}개의 샘플을 학습합니다.")
print("[진행] 모델 및 네트워크 초기화 중...")

model = JointModel().to(DEVICE)
D = PatchD().to(DEVICE)

if FREEZE_MASK:
    for p in model.mask_net.parameters():
        p.requires_grad=False
    print("[진행] 마스크 네트워크 가중치 동결 완료.")

opt_G = optim.Adam(model.parameters(),lr=LR_G)
opt_D = optim.Adam(D.parameters(),lr=LR_D)

bce = nn.BCEWithLogitsLoss()
bce_mask = nn.BCELoss()

print(f"[진행] 총 {EPOCHS} Epochs 학습을 시작합니다.")

for epoch in range(EPOCHS):

    loop = tqdm(loader, desc=f"Epoch [{epoch+1}/{EPOCHS}]")

    for pre,post,mask in loop:

        pre,post,mask = pre.to(DEVICE),post.to(DEVICE),mask.to(DEVICE)

        pred_mask,fake_post = model(pre,mask)

        loss_mask = bce_mask(pred_mask,mask) + dice(pred_mask,mask)

        real_input = torch.cat([pre,mask],1)

        real_pred = D(real_input,post)
        fake_pred = D(real_input,fake_post.detach())

        loss_D = (bce(real_pred,torch.ones_like(real_pred))+
                  bce(fake_pred,torch.zeros_like(fake_pred)))/2

        opt_D.zero_grad()
        loss_D.backward()
        opt_D.step()

        fake_pred = D(real_input,fake_post)

        loss_GAN = bce(fake_pred,torch.ones_like(fake_pred))

        loss_L1 = mask_weighted_l1(fake_post,post,mask,MASK_WEIGHT)*LAMBDA_L1

        loss_G = loss_GAN + loss_L1 + MASK_LOSS_WEIGHT*loss_mask

        opt_G.zero_grad()
        loss_G.backward()
        opt_G.step()

        loop.set_postfix(
            mask_loss=loss_mask.item(),
            G=loss_G.item(),
            D=loss_D.item()
        )

    # =========================
    # 50 Epoch마다 체크포인트 저장
    # =========================

    if (epoch+1) % 50 == 0:

        save_path = f"{CHECKPOINT_DIR}/joint_epoch_{epoch+1}.pt"

        torch.save({
            "epoch": epoch+1,
            "model_state_dict": model.state_dict(),
            "D_state_dict": D.state_dict(),
            "optimizer_G_state_dict": opt_G.state_dict(),
            "optimizer_D_state_dict": opt_D.state_dict()
        }, save_path)

        print(f"\n[저장] Epoch {epoch+1} 체크포인트 저장 완료 → {save_path}")

print("[완료] 모든 학습이 성공적으로 종료되었습니다.")

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
import cv2
import numpy as np
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

PATIENT_DIR = "./pix2pix_curated_withMask/TEST/100387B"
MODEL_PATH = "./joint_model_ckp2/joint_epoch_100.pt"


# =========================
# CLAHE
# =========================

def apply_clahe(img):

    img = np.array(img)

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8,8)
    )

    img = clahe.apply(img)

    return Image.fromarray(img)


# =========================
# Dataset
# =========================

class PatientDataset(Dataset):

    def __init__(self, patient_dir):

        self.samples = []

        self.tf = T.Compose([
            T.Resize((256,256)),
            T.ToTensor(),
            T.Normalize((0.5,), (0.5,))
        ])

        self.mask_tf = T.Compose([
            T.Resize((256,256)),
            T.ToTensor()
        ])

        pre_dir = os.path.join(patient_dir,"pre")
        post_dir = os.path.join(patient_dir,"post")
        mask_dir = os.path.join(patient_dir,"mask")

        for f in sorted(os.listdir(pre_dir)):

            pre = os.path.join(pre_dir,f)
            post = os.path.join(post_dir,f)
            mask = os.path.join(mask_dir,f)

            if os.path.exists(post) and os.path.exists(mask):

                self.samples.append((pre,post,mask))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self,idx):

        pre,post,mask = self.samples[idx]

        pre = Image.open(pre).convert("L")
        post = Image.open(post).convert("L")
        mask = Image.open(mask).convert("L")

        pre = apply_clahe(pre)
        post = apply_clahe(post)

        pre = self.tf(pre)
        post = self.tf(post)
        mask = self.mask_tf(mask)

        mask = (mask > 0.5).float()

        return pre,post,mask


# =========================
# UNet Block
# =========================

def CBR(in_c,out_c):

    return nn.Sequential(
        nn.Conv2d(in_c,out_c,3,1,1),
        nn.BatchNorm2d(out_c),
        nn.ReLU()
    )


# =========================
# Mask UNet
# =========================

class MaskUNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.d1 = nn.Sequential(CBR(1,64),CBR(64,64))
        self.p1 = nn.MaxPool2d(2)

        self.d2 = nn.Sequential(CBR(64,128),CBR(128,128))
        self.p2 = nn.MaxPool2d(2)

        self.d3 = nn.Sequential(CBR(128,256),CBR(256,256))
        self.p3 = nn.MaxPool2d(2)

        self.b = nn.Sequential(CBR(256,512),CBR(512,512))

        self.u3 = nn.ConvTranspose2d(512,256,2,2)
        self.c3 = nn.Sequential(CBR(512,256),CBR(256,256))

        self.u2 = nn.ConvTranspose2d(256,128,2,2)
        self.c2 = nn.Sequential(CBR(256,128),CBR(128,128))

        self.u1 = nn.ConvTranspose2d(128,64,2,2)
        self.c1 = nn.Sequential(CBR(128,64),CBR(64,64))

        self.out = nn.Conv2d(64,1,1)

    def forward(self,x):

        d1 = self.d1(x)
        d2 = self.d2(self.p1(d1))
        d3 = self.d3(self.p2(d2))

        b = self.b(self.p3(d3))

        u3 = self.u3(b)
        u3 = torch.cat([u3,d3],1)
        u3 = self.c3(u3)

        u2 = self.u2(u3)
        u2 = torch.cat([u2,d2],1)
        u2 = self.c2(u2)

        u1 = self.u1(u2)
        u1 = torch.cat([u1,d1],1)
        u1 = self.c1(u1)

        return torch.sigmoid(self.out(u1))


# =========================
# Generator (pix2pix style)
# =========================

class Generator(nn.Module):

    def __init__(self):

        super().__init__()

        self.model = nn.Sequential(

            nn.Conv2d(2,64,4,2,1),
            nn.LeakyReLU(0.2),

            nn.Conv2d(64,128,4,2,1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),

            nn.ConvTranspose2d(128,64,4,2,1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.ConvTranspose2d(64,1,4,2,1),
            nn.Tanh()
        )

    def forward(self,x):

        return self.model(x)


# =========================
# Joint Model
# =========================

class JointModel(nn.Module):

    def __init__(self):

        super().__init__()

        self.mask_net = MaskUNet()
        self.generator = Generator()

    def forward(self,pre):

        pred_mask = self.mask_net(pre)

        gen_input = torch.cat([pre,pred_mask],1)

        fake_post = self.generator(gen_input)

        return pred_mask,fake_post


# =========================
# Dataset Load
# =========================

dataset = PatientDataset(PATIENT_DIR)
loader = DataLoader(dataset,batch_size=1,shuffle=False)

print("slice 수:",len(dataset))


# =========================
# Model Load
# =========================

model = JointModel().to(DEVICE)

ckpt = torch.load(MODEL_PATH, map_location=DEVICE)

model.load_state_dict(ckpt["model_state_dict"])

model.eval()


# =========================
# Test + Visualization
# =========================

with torch.no_grad():

    for i,(pre,post,mask) in enumerate(loader):

        pre = pre.to(DEVICE)

        pred_mask,fake_post = model(pre)

        pre_img = pre.squeeze().cpu().numpy()
        gt_mask = mask.squeeze().cpu().numpy()
        pred_mask = pred_mask.squeeze().cpu().numpy()
        fake_post = fake_post.squeeze().cpu().numpy()
        gt_post = post.squeeze().cpu().numpy()

        pre_img = (pre_img+1)/2
        fake_post = (fake_post+1)/2
        gt_post = (gt_post+1)/2

        plt.figure(figsize=(18,4))

        titles = [
            "PRE",
            "GT_MASK",
            "PRED_MASK",
            "FAKE_POST",
            "GT_POST"
        ]

        imgs = [
            pre_img,
            gt_mask,
            pred_mask,
            fake_post,
            gt_post
        ]

        for j in range(5):

            plt.subplot(1,5,j+1)
            plt.title(titles[j])
            plt.imshow(imgs[j],cmap="gray")
            plt.axis("off")

        plt.show()

=================================================================================================



2중 UNET 구조로 변경 학습



=================================================================================================

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
from tqdm import tqdm
import cv2
import numpy as np

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATA_ROOT = "./pix2pix_curated_withMask_1-100"
SAVE_DIR = "./dual_unet_clahe_v2(1-100)"
os.makedirs(SAVE_DIR, exist_ok=True)

BATCH = 8
EPOCHS = 400

LR = 2e-4
LAMBDA_L1 = 100
MASK_WEIGHT = 15.0

# =========================
# CLAHE
# =========================
def apply_clahe(img):
    img = np.array(img)

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8,8)
    )

    img = clahe.apply(img)
    return Image.fromarray(img)

# =========================
# Dataset
# =========================
class FullDataset(Dataset):
    def __init__(self, root):
        self.samples = []

        self.tf = T.Compose([
            T.Resize((256,256)),
            T.ToTensor(),
            T.Normalize((0.5,), (0.5,))
        ])

        self.mask_tf = T.Compose([
            T.Resize((256,256)),
            T.ToTensor()
        ])

        for pid in sorted(os.listdir(root)):
            p = os.path.join(root, pid)
            pre_dir = os.path.join(p,"pre")
            post_dir = os.path.join(p,"post")
            mask_dir = os.path.join(p,"mask")

            if not os.path.isdir(pre_dir): continue

            for f in sorted(os.listdir(pre_dir)):
                pre = os.path.join(pre_dir,f)
                post = os.path.join(post_dir,f)
                mask = os.path.join(mask_dir,f)

                if os.path.exists(post) and os.path.exists(mask):
                    self.samples.append((pre,post,mask))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self,idx):
        pre,post,mask = self.samples[idx]

        pre = Image.open(pre).convert("L")
        post = Image.open(post).convert("L")
        mask = Image.open(mask).convert("L")

        #  CLAHE 적용
        pre = apply_clahe(pre)
        post = apply_clahe(post)

        pre = self.tf(pre)
        post = self.tf(post)
        mask = self.mask_tf(mask)

        mask = (mask > 0.5).float()

        return pre,post,mask

# =========================
# UNet (mask)
# =========================
def CBR(a,b):
    return nn.Sequential(
        nn.Conv2d(a,b,3,1,1),
        nn.BatchNorm2d(b),
        nn.ReLU()
    )

class MaskUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.d1 = nn.Sequential(CBR(1,64),CBR(64,64))
        self.p1 = nn.MaxPool2d(2)
        self.d2 = nn.Sequential(CBR(64,128),CBR(128,128))
        self.p2 = nn.MaxPool2d(2)
        self.d3 = nn.Sequential(CBR(128,256),CBR(256,256))
        self.p3 = nn.MaxPool2d(2)
        self.b = nn.Sequential(CBR(256,512),CBR(512,512))
        self.u3 = nn.ConvTranspose2d(512,256,2,2)
        self.c3 = nn.Sequential(CBR(512,256),CBR(256,256))
        self.u2 = nn.ConvTranspose2d(256,128,2,2)
        self.c2 = nn.Sequential(CBR(256,128),CBR(128,128))
        self.u1 = nn.ConvTranspose2d(128,64,2,2)
        self.c1 = nn.Sequential(CBR(128,64),CBR(64,64))
        self.out = nn.Conv2d(64,1,1)

    def forward(self,x):
        d1 = self.d1(x)
        d2 = self.d2(self.p1(d1))
        d3 = self.d3(self.p2(d2))
        b = self.b(self.p3(d3))
        u3 = self.u3(b); u3 = torch.cat([u3,d3],1); u3 = self.c3(u3)
        u2 = self.u2(u3); u2 = torch.cat([u2,d2],1); u2 = self.c2(u2)
        u1 = self.u1(u2); u1 = torch.cat([u1,d1],1); u1 = self.c1(u1)
        return torch.sigmoid(self.out(u1))

# =========================
# pix2pix Generator (U-Net)
# =========================
class UNetBlock(nn.Module):
    def __init__(self, in_ch, out_ch, submodule=None, innermost=False, outermost=False):
        super().__init__()
        self.outermost = outermost

        downconv = nn.Conv2d(in_ch, out_ch, 4, 2, 1, bias=False)
        downrelu = nn.LeakyReLU(0.2, True)
        downnorm = nn.BatchNorm2d(out_ch)
        uprelu = nn.ReLU(True)

        if outermost:
            upconv = nn.ConvTranspose2d(out_ch * 2, 1, 4, 2, 1)
            model = [downconv] + [submodule] + [uprelu, upconv, nn.Tanh()]
        elif innermost:
            upconv = nn.ConvTranspose2d(out_ch, in_ch, 4, 2, 1, bias=False)
            model = [downrelu, downconv] + [uprelu, upconv, nn.BatchNorm2d(in_ch)]
        else:
            upconv = nn.ConvTranspose2d(out_ch * 2, in_ch, 4, 2, 1, bias=False)
            model = [downrelu, downconv, downnorm] + [submodule] + [uprelu, upconv, nn.BatchNorm2d(in_ch)]

        self.model = nn.Sequential(*model)

    def forward(self, x):
        if self.outermost:
            return self.model(x)
        else:
            return torch.cat([x, self.model(x)], 1)

class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        nf = 64
        b8 = UNetBlock(nf*8, nf*8, innermost=True)
        b7 = UNetBlock(nf*8, nf*8, submodule=b8)
        b6 = UNetBlock(nf*8, nf*8, submodule=b7)
        b5 = UNetBlock(nf*8, nf*8, submodule=b6)
        b4 = UNetBlock(nf*4, nf*8, submodule=b5)
        b3 = UNetBlock(nf*2, nf*4, submodule=b4)
        b2 = UNetBlock(nf, nf*2, submodule=b3)
        self.model = UNetBlock(2, nf, submodule=b2, outermost=True)

    def forward(self,x):
        return self.model(x)

# =========================
# Discriminator
# =========================
class PatchD(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3,64,4,2,1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64,128,4,2,1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.Conv2d(128,1,4,1,1)
        )

    def forward(self,x,y):
        return self.model(torch.cat([x,y],1))

# =========================
# Joint Model
# =========================
class DualModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.mask_net = MaskUNet()
        self.gen = Generator()

    def forward(self, pre, mask_gt=None):
        pred_mask = self.mask_net(pre)

        if self.training:
            mask_in = mask_gt
        else:
            mask_in = pred_mask

        gen_input = torch.cat([pre,mask_in],1)
        fake_post = self.gen(gen_input)

        return pred_mask, fake_post

# =========================
# Loss
# =========================
bce = nn.BCEWithLogitsLoss()
bce_mask = nn.BCELoss()

def dice(pred,target):
    smooth=1e-5
    p=pred.view(-1)
    t=target.view(-1)
    inter=(p*t).sum()
    return 1-(2*inter+smooth)/(p.sum()+t.sum()+smooth)

def mask_l1(pred,target,mask):
    loss = torch.abs(pred-target)
    weight = 1 + mask*(MASK_WEIGHT-1)
    return torch.mean(loss*weight)

# =========================
# Train
# =========================
dataset = FullDataset(f"{DATA_ROOT}/TRAIN")
loader = DataLoader(dataset,batch_size=BATCH,shuffle=True)

model = DualModel().to(DEVICE)
D = PatchD().to(DEVICE)

opt_G = optim.Adam(model.parameters(), lr=LR, betas=(0.5,0.999))
opt_D = optim.Adam(D.parameters(), lr=LR, betas=(0.5,0.999))

for epoch in range(EPOCHS):

    loop = tqdm(loader)

    for pre,post,mask in loop:

        pre,post,mask = pre.to(DEVICE),post.to(DEVICE),mask.to(DEVICE)

        pred_mask,fake = model(pre,mask)

        loss_mask = bce_mask(pred_mask,mask) + dice(pred_mask,mask)

        cond = torch.cat([pre,mask],1)
        real_pred = D(cond,post)
        fake_pred = D(cond,fake.detach())

        loss_D = (bce(real_pred,torch.ones_like(real_pred)) +
                  bce(fake_pred,torch.zeros_like(fake_pred))) * 0.5

        opt_D.zero_grad()
        loss_D.backward()
        opt_D.step()

        fake_pred = D(cond,fake)

        loss_GAN = bce(fake_pred,torch.ones_like(fake_pred))
        loss_L1 = mask_l1(fake,post,mask) * LAMBDA_L1

        loss_G = loss_GAN + loss_L1 + loss_mask

        opt_G.zero_grad()
        loss_G.backward()
        opt_G.step()

        loop.set_description(f"Epoch {epoch+1}")
        loop.set_postfix(G=loss_G.item(), D=loss_D.item())

    if (epoch+1) % 50 == 0:
        path = f"{SAVE_DIR}/dual_epoch_{epoch+1}.pt"
        torch.save({
            "model": model.state_dict(),
            "D": D.state_dict()
        }, path)
        print(f"\n 저장 완료: {path}")

print("학습 완료")

In [ ]:
2중 UNET 테스트

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
import matplotlib.pyplot as plt
import cv2
import numpy as np

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TEST_DIR = "./pix2pix_curated_withMask_1-100/TEST/100285A"
MODEL_PATH = "./dual_unet_clahe_v2/dual_epoch_200.pt"

#  추가 (저장 폴더)
SAVE_DIR = "./results"
os.makedirs(SAVE_DIR, exist_ok=True)


# =========================
# CLAHE
# =========================
def apply_clahe(img):
    img = np.array(img)

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8,8)
    )

    img = clahe.apply(img)

    return Image.fromarray(img)


# =========================
# Dataset
# =========================
class TestDataset(Dataset):
    def __init__(self, root):
        self.samples = []

        self.tf = T.Compose([
            T.Resize((256,256)),
            T.ToTensor(),
            T.Normalize((0.5,), (0.5,))
        ])

        self.mask_tf = T.Compose([
            T.Resize((256,256)),
            T.ToTensor()
        ])

        pre_dir = os.path.join(root,"pre")
        post_dir = os.path.join(root,"post")
        mask_dir = os.path.join(root,"mask")

        for f in sorted(os.listdir(pre_dir)):
            pre = os.path.join(pre_dir,f)
            post = os.path.join(post_dir,f)
            mask = os.path.join(mask_dir,f)

            if os.path.exists(post) and os.path.exists(mask):
                self.samples.append((pre,post,mask))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self,idx):
        pre,post,mask = self.samples[idx]

        pre = Image.open(pre).convert("L")
        post = Image.open(post).convert("L")
        mask = Image.open(mask).convert("L")

        # CLAHE
        pre = apply_clahe(pre)
        post = apply_clahe(post)

        pre = self.tf(pre)
        post = self.tf(post)
        mask = self.mask_tf(mask)

        mask = (mask > 0.5).float()

        return pre,post,mask


# =========================
# 모델 정의 (학습과 동일)
# =========================
def CBR(a,b):
    return nn.Sequential(
        nn.Conv2d(a,b,3,1,1),
        nn.BatchNorm2d(b),
        nn.ReLU()
    )

class MaskUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.d1 = nn.Sequential(CBR(1,64),CBR(64,64))
        self.p1 = nn.MaxPool2d(2)
        self.d2 = nn.Sequential(CBR(64,128),CBR(128,128))
        self.p2 = nn.MaxPool2d(2)
        self.d3 = nn.Sequential(CBR(128,256),CBR(256,256))
        self.p3 = nn.MaxPool2d(2)
        self.b = nn.Sequential(CBR(256,512),CBR(512,512))
        self.u3 = nn.ConvTranspose2d(512,256,2,2)
        self.c3 = nn.Sequential(CBR(512,256),CBR(256,256))
        self.u2 = nn.ConvTranspose2d(256,128,2,2)
        self.c2 = nn.Sequential(CBR(256,128),CBR(128,128))
        self.u1 = nn.ConvTranspose2d(128,64,2,2)
        self.c1 = nn.Sequential(CBR(128,64),CBR(64,64))
        self.out = nn.Conv2d(64,1,1)

    def forward(self,x):
        d1 = self.d1(x)
        d2 = self.d2(self.p1(d1))
        d3 = self.d3(self.p2(d2))
        b = self.b(self.p3(d3))
        u3 = self.u3(b); u3 = torch.cat([u3,d3],1); u3 = self.c3(u3)
        u2 = self.u2(u3); u2 = torch.cat([u2,d2],1); u2 = self.c2(u2)
        u1 = self.u1(u2); u1 = torch.cat([u1,d1],1); u1 = self.c1(u1)
        return torch.sigmoid(self.out(u1))


class UNetBlock(nn.Module):
    def __init__(self, in_ch, out_ch, submodule=None, innermost=False, outermost=False):
        super().__init__()
        self.outermost = outermost

        downconv = nn.Conv2d(in_ch, out_ch, 4, 2, 1, bias=False)
        downrelu = nn.LeakyReLU(0.2, True)
        downnorm = nn.BatchNorm2d(out_ch)
        uprelu = nn.ReLU(True)

        if outermost:
            upconv = nn.ConvTranspose2d(out_ch * 2, 1, 4, 2, 1)
            model = [downconv] + [submodule] + [uprelu, upconv, nn.Tanh()]
        elif innermost:
            upconv = nn.ConvTranspose2d(out_ch, in_ch, 4, 2, 1, bias=False)
            model = [downrelu, downconv] + [uprelu, upconv, nn.BatchNorm2d(in_ch)]
        else:
            upconv = nn.ConvTranspose2d(out_ch * 2, in_ch, 4, 2, 1, bias=False)
            model = [downrelu, downconv, downnorm] + [submodule] + [uprelu, upconv, nn.BatchNorm2d(in_ch)]

        self.model = nn.Sequential(*model)

    def forward(self, x):
        if self.outermost:
            return self.model(x)
        else:
            return torch.cat([x, self.model(x)], 1)


class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        nf = 64
        b8 = UNetBlock(nf*8, nf*8, innermost=True)
        b7 = UNetBlock(nf*8, nf*8, submodule=b8)
        b6 = UNetBlock(nf*8, nf*8, submodule=b7)
        b5 = UNetBlock(nf*8, nf*8, submodule=b6)
        b4 = UNetBlock(nf*4, nf*8, submodule=b5)
        b3 = UNetBlock(nf*2, nf*4, submodule=b4)
        b2 = UNetBlock(nf, nf*2, submodule=b3)
        self.model = UNetBlock(2, nf, submodule=b2, outermost=True)

    def forward(self,x):
        return self.model(x)


class DualModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.mask_net = MaskUNet()
        self.gen = Generator()

    def forward(self, pre):
        pred_mask = self.mask_net(pre)
        gen_input = torch.cat([pre,pred_mask],1)
        fake_post = self.gen(gen_input)
        return pred_mask, fake_post


# =========================
# 로드
# =========================
dataset = TestDataset(TEST_DIR)
loader = DataLoader(dataset,batch_size=1,shuffle=False)

model = DualModel().to(DEVICE)

ckpt = torch.load(MODEL_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model"])

model.eval()


# =========================
# 테스트 + 시각화
# =========================
with torch.no_grad():
    for i,(pre,post,mask) in enumerate(loader):

        pre = pre.to(DEVICE)

        pred_mask, fake_post = model(pre)

        pre_img = pre.squeeze().cpu().numpy()
        gt_mask = mask.squeeze().cpu().numpy()
        pred_mask = pred_mask.squeeze().cpu().numpy()
        fake_post = fake_post.squeeze().cpu().numpy()
        gt_post = post.squeeze().cpu().numpy()

        pre_img = (pre_img + 1) / 2
        fake_post = (fake_post + 1) / 2
        gt_post = (gt_post + 1) / 2

        plt.figure(figsize=(18,4))

        titles = ["PRE","GT_MASK","PRED_MASK","FAKE_POST","GT_POST"]
        imgs = [pre_img, gt_mask, pred_mask, fake_post, gt_post]

        for j in range(5):
            plt.subplot(1,5,j+1)
            plt.title(titles[j])
            plt.imshow(imgs[j], cmap="gray")
            plt.axis("off")

        # =========================
        # 초기화 (루프 밖에 한 번만)
        # =========================
        if i == 0:
            all_rows = []
        
        PAD = 5
        
        h, w = pre_img.shape
        v_pad = np.ones((h, PAD))  # 흰색 공백
        
        # =========================
        # 5개 이미지 + padding
        # =========================
        combined = np.concatenate([
            pre_img, v_pad,
            gt_mask, v_pad,
            pred_mask, v_pad,
            fake_post, v_pad,
            gt_post
        ], axis=1)
        
        # 0~1 → 0~255
        combined = (combined * 255).astype(np.uint8)
        combined = cv2.cvtColor(combined, cv2.COLOR_GRAY2BGR)
        
        # =========================
        # 텍스트 (슬라이스 + 라벨)
        # =========================
        cv2.putText(combined, f"Slice {i}", (10, 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)
        
        labels = ["PRE", "GT_MASK", "PRED_MASK", "FAKE_POST", "GT_POST"]
        step = w + PAD
        
        for idx, label in enumerate(labels):
            x = idx * step + 5
            cv2.putText(combined, label, (x, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255,200,0), 1)
        
        # =========================
        # 리스트에 누적
        # =========================
        all_rows.append(combined)
        
        
        # =========================
        # 마지막에 한 번만 저장
        # =========================
        if i == len(loader) - 1:
        
            row_h, row_w, _ = all_rows[0].shape
            h_pad = np.ones((PAD, row_w, 3), dtype=np.uint8) * 255
        
            final_list = []
            for r in all_rows:
                final_list.append(r)
                final_list.append(h_pad)
        
            final_img = np.concatenate(final_list[:-1], axis=0)
        
            cv2.imwrite(
                os.path.join(SAVE_DIR, "final_result.png"),
                final_img
            )

        plt.show()

In [ ]:
#이후 마스크 생성에 대한 성능 변화와 FKAE_POST 생성 퀄리티 비교 확인 필요함
#성능지표에 대한 문제 해결해야함 

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
import cv2
import numpy as np
from skimage.metrics import structural_similarity as ssim

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TEST_DIR = "./pix2pix_curated_withMask_1-100/TEST/100324A"
MODEL_PATH = "./pix2pix_checkpoints_v2/G_epoch_200.pt"


# =========================
# CLAHE
# =========================
def apply_clahe(img):
    img = np.array(img)

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8,8)
    )

    img = clahe.apply(img)

    return Image.fromarray(img)


# =========================
# Dataset
# =========================
class TestDataset(Dataset):
    def __init__(self, root):
        self.samples = []

        self.tf = T.Compose([
            T.Resize((256,256)),
            T.ToTensor(),
            T.Normalize((0.5,), (0.5,))
        ])

        self.mask_tf = T.Compose([
            T.Resize((256,256)),
            T.ToTensor()
        ])

        pre_dir = os.path.join(root,"pre")
        post_dir = os.path.join(root,"post")
        mask_dir = os.path.join(root,"mask")

        for f in sorted(os.listdir(pre_dir)):
            pre = os.path.join(pre_dir,f)
            post = os.path.join(post_dir,f)
            mask = os.path.join(mask_dir,f)

            if os.path.exists(post) and os.path.exists(mask):
                self.samples.append((pre,post,mask))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self,idx):
        pre,post,mask = self.samples[idx]

        pre = Image.open(pre).convert("L")
        post = Image.open(post).convert("L")
        mask = Image.open(mask).convert("L")

        # CLAHE
        pre = apply_clahe(pre)
        post = apply_clahe(post)

        pre = self.tf(pre)
        post = self.tf(post)
        mask = self.mask_tf(mask)

        mask = (mask > 0.5).float()

        return pre,post,mask


# =========================
# 모델 정의
# =========================
def CBR(a,b):
    return nn.Sequential(
        nn.Conv2d(a,b,3,1,1),
        nn.BatchNorm2d(b),
        nn.ReLU()
    )

class MaskUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.d1 = nn.Sequential(CBR(1,64),CBR(64,64))
        self.p1 = nn.MaxPool2d(2)
        self.d2 = nn.Sequential(CBR(64,128),CBR(128,128))
        self.p2 = nn.MaxPool2d(2)
        self.d3 = nn.Sequential(CBR(128,256),CBR(256,256))
        self.p3 = nn.MaxPool2d(2)
        self.b = nn.Sequential(CBR(256,512),CBR(512,512))
        self.u3 = nn.ConvTranspose2d(512,256,2,2)
        self.c3 = nn.Sequential(CBR(512,256),CBR(256,256))
        self.u2 = nn.ConvTranspose2d(256,128,2,2)
        self.c2 = nn.Sequential(CBR(256,128),CBR(128,128))
        self.u1 = nn.ConvTranspose2d(128,64,2,2)
        self.c1 = nn.Sequential(CBR(128,64),CBR(64,64))
        self.out = nn.Conv2d(64,1,1)

    def forward(self,x):
        d1 = self.d1(x)
        d2 = self.d2(self.p1(d1))
        d3 = self.d3(self.p2(d2))
        b = self.b(self.p3(d3))
        u3 = self.u3(b); u3 = torch.cat([u3,d3],1); u3 = self.c3(u3)
        u2 = self.u2(u3); u2 = torch.cat([u2,d2],1); u2 = self.c2(u2)
        u1 = self.u1(u2); u1 = torch.cat([u1,d1],1); u1 = self.c1(u1)
        return torch.sigmoid(self.out(u1))


class UNetBlock(nn.Module):
    def __init__(self, in_ch, out_ch, submodule=None, innermost=False, outermost=False):
        super().__init__()
        self.outermost = outermost

        downconv = nn.Conv2d(in_ch, out_ch, 4, 2, 1, bias=False)
        downrelu = nn.LeakyReLU(0.2, True)
        downnorm = nn.BatchNorm2d(out_ch)
        uprelu = nn.ReLU(True)

        if outermost:
            upconv = nn.ConvTranspose2d(out_ch * 2, 1, 4, 2, 1)
            model = [downconv] + [submodule] + [uprelu, upconv, nn.Tanh()]
        elif innermost:
            upconv = nn.ConvTranspose2d(out_ch, in_ch, 4, 2, 1, bias=False)
            model = [downrelu, downconv] + [uprelu, upconv, nn.BatchNorm2d(in_ch)]
        else:
            upconv = nn.ConvTranspose2d(out_ch * 2, in_ch, 4, 2, 1, bias=False)
            model = [downrelu, downconv, downnorm] + [submodule] + [uprelu, upconv, nn.BatchNorm2d(in_ch)]

        self.model = nn.Sequential(*model)

    def forward(self, x):
        if self.outermost:
            return self.model(x)
        else:
            return torch.cat([x, self.model(x)], 1)


class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        nf = 64
        b8 = UNetBlock(nf*8, nf*8, innermost=True)
        b7 = UNetBlock(nf*8, nf*8, submodule=b8)
        b6 = UNetBlock(nf*8, nf*8, submodule=b7)
        b5 = UNetBlock(nf*8, nf*8, submodule=b6)
        b4 = UNetBlock(nf*4, nf*8, submodule=b5)
        b3 = UNetBlock(nf*2, nf*4, submodule=b4)
        b2 = UNetBlock(nf, nf*2, submodule=b3)
        self.model = UNetBlock(2, nf, submodule=b2, outermost=True)

    def forward(self,x):
        return self.model(x)


class DualModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.mask_net = MaskUNet()
        self.gen = Generator()

    def forward(self, pre):
        pred_mask = self.mask_net(pre)
        gen_input = torch.cat([pre,pred_mask],1)
        fake_post = self.gen(gen_input)
        return pred_mask, fake_post


# =========================
# 로드
# =========================
dataset = TestDataset(TEST_DIR)
loader = DataLoader(dataset,batch_size=1,shuffle=False)

model = DualModel().to(DEVICE)

ckpt = torch.load(MODEL_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model"])
model.eval()


# =========================
# SSIM 평가
# =========================
ssim_list = []

with torch.no_grad():
    for i,(pre,post,mask) in enumerate(loader):

        pre = pre.to(DEVICE)

        _, fake_post = model(pre)

        fake_post = fake_post.squeeze().cpu().numpy()
        gt_post = post.squeeze().cpu().numpy()

        # [-1,1] → [0,1]
        fake_post = (fake_post + 1) / 2
        gt_post = (gt_post + 1) / 2

        score = ssim(fake_post, gt_post, data_range=1.0)
        ssim_list.append(score)

        print(f"[Slice {i}] SSIM: {score:.4f}")


print("\n===== FINAL RESULT =====")
print(f"Mean SSIM: {np.mean(ssim_list):.4f}")
print(f"Std SSIM : {np.std(ssim_list):.4f}")